# NB09 — Accra Flood Intelligence Platform
## Odaw River Basin & Korle Lagoon, Ghana

**The problem:** Accra, Ghana, floods almost every rainy season — often fatally.
The June 2015 flood killed over 150 people when floodwater carried fuel from a
nearby pipe to a filling station where residents had taken shelter. A May 2025 storm
killed four people and displaced more than 3,000. Heavy flooding recurred again in
June 2026. The Odaw River Basin holds an estimated 2.5 million people; the World
Bank committed US$150 million to drainage infrastructure (GARID project), but
infrastructure construction takes years. Communities need real-time warning now.

**What this notebook builds:**

| Module | Output | Decision value |
|---|---|---|
| Historical SAR flood frequency | 20+ scene composite flood probability map | Where flooding is **recurrent**, not just where it happened once |
| Active inundation mapping | Current flood extent from latest SAR pass | Real-time: which streets are underwater **right now** |
| Prithvi-based segmentation | High-accuracy flood mask (Sen1Floods11 fine-tune) | Better than simple threshold in vegetated/urban margins |
| Risk zone classification | 4-tier (Safe/Watch/Warning/Emergency) per ward | Operationally actionable: which wards trigger which response |
| FloodWatch Ghana output | GeoJSON alert zones + severity scores | Direct feed into the IoT alerting dashboard |

**Sensor strategy:** Sentinel-1 SAR first — cloud-independent and available within
6 hours of a new pass, regardless of the dense cloud cover that Accra's rainy season
brings. Sentinel-2 optical used for land cover context and calibration only.

**Study areas:**
- **Odaw River Basin** (primary): covers Alajo, Agbogbloshie, Kaneshie, Nima, Accra CBD
- **Korle Lagoon** (secondary): downstream receiving basin, chronic flooding

**Rainy seasons:** Major (Apr–Jul), Minor (Sep–Nov) — two flood seasons per year


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PyGeoVision v2.0 — Notebook Setup Cell                                    ║
# ║  Run this cell FIRST. Defines all imports and helper functions.             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import warnings; warnings.filterwarnings('ignore')
import pathlib, json, math, copy, time, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import uniform_filter

# -- PyTorch (optional -- training cells skipped if absent) --
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    HAS_TORCH = True
except ImportError:
    torch = nn = F = DataLoader = TensorDataset = None
    DEVICE = 'cpu'; HAS_TORCH = False
    print("PyTorch not installed -- training cells will be skipped")
    print("  pip install torch torchvision")
# ── Rasterio ───────────────────────────────────────────────────────────────────
import rasterio
import rasterio.transform as rt
from rasterio.windows import from_bounds

# ── PyGeoVision — core client ─────────────────────────────────────────────────
import pygeovision as pgv
from pygeovision import PyGeoVision
client = PyGeoVision()

# ── PyGeoVision — models ──────────────────────────────────────────────────────
from pygeovision.models import get_model
from pygeovision.models.change_detection.changeformer import ChangeFormer   # NOT ChangeDetection
from pygeovision.models.foundation.prithvi import (
    PrithviTasks, Prithvi,
    validate_prithvi_input,
    map_bands, normalise_hls,
    HLS_SCALE_FACTOR,
    LAND_COVER_CLASSES, LAND_COVER_PALETTE, CROP_CLASSES,
    SENTINEL2_CANONICAL, LANDSAT_CANONICAL,
    _spectral_land_cover, _spectral_flood,
    _spectral_burn_scar, _spectral_crop_mapping,
)
from pygeovision.models.foundation.dinov3 import DINOv3Backbone, get_transform, CHMv2Model
from pygeovision.inference.tiled import TiledInference

# ── PyGeoVision — training (requires PyTorch) ───────────────────────
if HAS_TORCH:
    from pygeovision.ai.training.losses    import DiceFocalLoss
    from pygeovision.training.metrics      import SegmentationMetrics
    from pygeovision.training.checkpoint   import CheckpointManager
    from pygeovision.training.callbacks    import EarlyStopping
    from pygeovision.training.experiment   import ExperimentTracker
    from pygeovision.training.mixed_precision import MixedPrecisionManager
    try:
        from pygeovision.ai.training.export import ModelExporter
    except Exception:
        ModelExporter = None
else:
    DiceFocalLoss = SegmentationMetrics = CheckpointManager = None
    EarlyStopping = ExperimentTracker = MixedPrecisionManager = ModelExporter = None
# ── PyGeoVision — data & datasets ────────────────────────────────────────────
from pygeovision.datasets.registry import DatasetRegistry
from pygeovision.datasets.loader   import DatasetLoader
from pygeovision.datasets.analysis import DatasetAnalyzer



# ── PyGeoVision — SAR bug-fix preprocessing modules (Jul 2026) ──────────────
from pygeovision.data.processors.sar import (
    verify_sar_downloads, validate_sar_georeference,
    despeckle_sar, linear_to_db, normalise_sar_for_ai,
    clip_sar_to_bbox, SARPreprocessor,
)
from pygeovision.data.validators.georeference import (
    check_download_complete, reproject_bbox_to_raster_crs,
)
from pygeovision.models.adapters.sar_channel_manager import (
    sar_to_hls_6ch, sar_to_pseudo_rgb, coregister_sar_pair, validate_sar_ai_input,
)
from pygeovision.models.adapters.sar_prithvi import SARPrithviAdapter, Sen1Floods11Config
from pygeovision.models.adapters.sar_dinov3  import SARDINOv3Adapter, SARSpeckleAugmentation



07:50:39 INFO [      engine] PyGeoFetch ready


2026-07-13 07:50:39,281 [INFO] pygeofetch.core.engine: PyGeoFetch ready


07:50:39 INFO [      engine] PyGeoFetch ready


2026-07-13 07:50:39,283 [INFO] pygeofetch.core.engine: PyGeoFetch ready
2026-07-13 07:50:39,284 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v1.0 Python API active — using native search/download
2026-07-13 07:50:39,284 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v2.0 processing engine detected — using native preprocessing
2026-07-13 07:50:39,289 [INFO] pygeovision.models.registry: Model registry: 119 architectures loaded


In [3]:
reg    = DatasetRegistry()
loader = DatasetLoader()

CHIP_SIZE       = 256     # spatial chip dimension for training patches
TRAIN_VAL_SPLIT = 0.80    # 80% train / 20% validation
BENCH_EPOCHS    = 20      # epochs for benchmark-dataset training
DOMAIN_EPOCHS   = 10      # epochs for domain fine-tuning (warm-start)
FT_LR_HEAD      = 1e-4   # learning rate for task head
FT_LR_DOMAIN    = 5e-5   # lower LR for domain fine-tuning
BATCH_SIZE      = 6       # training batch size


In [4]:

# ── PyGeoSpace — interactive maps (graceful fallback if not installed) ─────────
try:
    from pygeospace.folium import Map as PgsMap, RasterLayer, VectorLayer
    HAS_PYGEOSPACE = True
    print("PyGeoSpace: OK interactive maps enabled")
except ImportError:
    PgsMap = RasterLayer = VectorLayer = None
    HAS_PYGEOSPACE = False
    print("PyGeoSpace: not installed -- matplotlib fallback active (all maps work)")
    print("  pip install pygeospace   to enable interactive folium maps")

# ── Map helper functions (work with or without PyGeoSpace) ────────────────────
def show_raster_map(arr, bounds, title="", cmap='tab20', vmin=None, vmax=None,
                    save_path=None, colorbar_label="", legend_dict=None):
    """Raster map — PyGeoSpace (interactive) or matplotlib (static fallback)."""
    if HAS_PYGEOSPACE:
        lat_c=(bounds[1]+bounds[3])/2; lon_c=(bounds[0]+bounds[2])/2
        m = PgsMap(location=(lat_c,lon_c), zoom_start=11,
                   basemap='esri-satellite', title=title)
        m.add_layer(RasterLayer(arr.astype('float32'),
                                bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                cmap=cmap, opacity=0.75,
                                vmin=vmin or float(arr.min()),
                                vmax=vmax or float(arr.max())))
        if colorbar_label:
            m.add_colorbar(vmin or float(arr.min()), vmax or float(arr.max()),
                           cmap=cmap, caption=colorbar_label)
        if legend_dict: m.add_legend(legend_dict)
        m.add_title(title).add_scale()
        if save_path: m.save(str(save_path))
        m.show()
        return m
    else:
        fig, ax = plt.subplots(figsize=(10,8))
        im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=14, pad=10); ax.axis('off')
        plt.colorbar(im, ax=ax, label=colorbar_label or cmap, fraction=0.03)
        if legend_dict:
            patches = [mpatches.Patch(color=c, label=n) for n,c in legend_dict.items()]
            ax.legend(handles=patches, loc='lower right', fontsize=9,
                      framealpha=0.85, borderpad=0.6)
        plt.tight_layout()
        if save_path:
            p = str(save_path).replace('.html','.png')
            plt.savefig(p, dpi=150, bbox_inches='tight'); print(f"Saved: {p}")
        plt.show(); return fig

def show_split_map(arr_l, arr_r, bounds, title="",
                   left_label="Pre-trained", right_label="Fine-tuned",
                   cmap='tab20', save_path=None, legend_dict=None):
    """Split comparison — PyGeoSpace drag-divider or side-by-side matplotlib."""
    if HAS_PYGEOSPACE:
        lat_c=(bounds[1]+bounds[3])/2; lon_c=(bounds[0]+bounds[2])/2
        m = PgsMap(location=(lat_c,lon_c), zoom_start=11, basemap='esri-satellite')
        m.split_map(
            left_layer =RasterLayer(arr_l.astype('float32'),
                                    bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                    cmap=cmap, opacity=0.82),
            right_layer=RasterLayer(arr_r.astype('float32'),
                                    bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                    cmap=cmap, opacity=0.82),
            left_label=left_label, right_label=right_label)
        m.add_title(title, subtitle="Drag divider ← → to compare")
        if legend_dict: m.add_legend(legend_dict)
        m.add_full_ui()
        if save_path: m.save(str(save_path))
        m.show(); return m
    else:
        fig, axes = plt.subplots(1,2,figsize=(16,7), constrained_layout=True)
        axes[0].imshow(arr_l, cmap=cmap); axes[0].set_title(left_label, fontsize=13); axes[0].axis('off')
        axes[1].imshow(arr_r, cmap=cmap); axes[1].set_title(right_label,fontsize=13); axes[1].axis('off')
        fig.suptitle(title, fontsize=15)
        if save_path:
            p = str(save_path).replace('.html','.png')
            plt.savefig(p, dpi=150, bbox_inches='tight'); print(f"Saved: {p}")
        plt.show(); return fig

def show_vector_map(vector_path, raster_arr=None, bounds=None, title="",
                    fill_color='#ff4400', save_path=None):
    """Vector polygon map with optional raster background."""
    if HAS_PYGEOSPACE and bounds:
        lat_c=(bounds[1]+bounds[3])/2; lon_c=(bounds[0]+bounds[2])/2
        m = PgsMap(location=(lat_c,lon_c), zoom_start=11,
                   basemap='esri-satellite', title=title)
        if raster_arr is not None:
            m.add_layer(RasterLayer(raster_arr.astype('float32'),
                                    bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                    cmap='tab20', opacity=0.55))
        if pathlib.Path(str(vector_path)).exists():
            m.add_layer(VectorLayer(str(vector_path),
                                    fill_color=fill_color, fill_opacity=0.45, weight=1.5))
        m.add_title(title).add_scale()
        if save_path: m.save(str(save_path))
        m.show(); return m
    else:
        print(f"[map] {title}")
        if pathlib.Path(str(vector_path)).exists():
            with open(vector_path) as f: gj = json.load(f)
            print(f"  {len(gj.get('features',[]))} features in {pathlib.Path(vector_path).name}")
        return None

PyGeoSpace: OK interactive maps enabled


In [5]:
# ── Training helper functions ─────────────────────────────────────────────────
def chips_from_raster(path, chip_size=256, max_chips=120):
    """Slice a preprocessed raster into (N, C, chip_size, chip_size) chips."""
    with rasterio.open(str(path)) as src:
        data = src.read().astype('float32')
        bounds = src.bounds
    C, H, W = data.shape
    chips = []
    for y in range(0, H - chip_size + 1, chip_size):
        for x in range(0, W - chip_size + 1, chip_size):
            chip = data[:, y:y+chip_size, x:x+chip_size]
            if not np.isnan(chip).all():
                chips.append(chip)
            if len(chips) >= max_chips:
                return np.stack(chips), bounds
    if chips:
        return np.stack(chips), bounds
    return np.zeros((1, C, chip_size, chip_size), 'float32'), bounds

def aug_chip(chip):
    """Random flip + rotate + noise augmentation for geospatial chips."""
    if np.random.rand() > 0.5: chip = chip[:, ::-1, :].copy()
    if np.random.rand() > 0.5: chip = chip[:, :, ::-1].copy()
    k = np.random.randint(4)
    if k: chip = np.rot90(chip, k, axes=(1,2)).copy()
    chip = np.clip(chip + np.random.normal(0, 0.005, chip.shape).astype('float32'), 0, 1)
    return chip

def make_loaders(chips, labels, batch_size=6, split=0.80):
    """Build train/val DataLoaders with augmentation on the training split."""
    n   = len(chips)
    idx = np.random.permutation(n)
    s   = int(n * split)
    tr_chips = np.stack([aug_chip(chips[i]) for i in idx[:s]])
    va_chips  = chips[idx[s:]]
    tr = TensorDataset(torch.tensor(tr_chips),   torch.tensor(labels[idx[:s]]))
    va = TensorDataset(torch.tensor(va_chips),    torch.tensor(labels[idx[s:]]))
    return (DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=0),
            DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=0))

def run_epochs(model, tr_loader, va_loader, n_epochs, criterion,
               lr=1e-4, n_classes=2, out_dir='./ckpt', label='model'):
    """Standard training loop with cosine LR, mixed precision, and checkpointing."""
    out_dir = pathlib.Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    opt   = torch.optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()),
                               lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr*0.01)
    mp    = MixedPrecisionManager(precision='auto', enabled=(DEVICE=='cuda'))
    ckpt  = CheckpointManager(str(out_dir), monitor='val_miou', mode='max')
    best  = 0.0
    for ep in range(n_epochs):
        model.train(); ep_loss = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            with mp.autocast():
                logits = model(xb)
                loss   = criterion(logits, yb)
            mp.scale_and_step(loss, opt)
            ep_loss.append(loss.item())
        model.eval(); vp_all, vl_all = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                logits = model(xb.to(DEVICE))
                vp_all.append(logits.argmax(1).cpu())
                vl_all.append(yb)
        vp = torch.cat(vp_all).numpy(); vl = torch.cat(vl_all).numpy()
        ious = []
        for c in range(n_classes):
            tp = ((vp==c)&(vl==c)).sum(); fp = ((vp==c)&(vl!=c)).sum(); fn = ((vp!=c)&(vl==c)).sum()
            ious.append(float(tp/(tp+fp+fn+1e-8)))
        miou = float(np.mean(ious))
        sched.step()
        if miou > best:
            best = miou
            ckpt.save(ep, model, opt, metrics={'val_miou': miou})
        if (ep+1) % max(1, n_epochs//5) == 0:
            print(f"  Ep {ep+1:3d}/{n_epochs}  loss={np.mean(ep_loss):.4f}"
                  f"  val_mIoU={miou*100:.1f}%  best={best*100:.1f}%")
    return best, ckpt

def run_epochs_bitemporal(model, tr_loader, va_loader, n_epochs, criterion,
                           lr=1e-4, n_classes=2, out_dir='./ckpt', label='model'):
    """Training loop for bi-temporal models (ChangeFormer: takes 2 inputs)."""
    out_dir = pathlib.Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    ckpt  = CheckpointManager(str(out_dir), monitor='val_miou', mode='max')
    best  = 0.0
    n_bands_per = None   # inferred on first batch
    for ep in range(n_epochs):
        model.train(); ep_loss = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if n_bands_per is None: n_bands_per = xb.shape[1] // 2
            opt.zero_grad()
            logits = model(xb[:, :n_bands_per, ...], xb[:, n_bands_per:, ...])
            loss   = criterion(logits, yb)
            loss.backward(); opt.step()
            ep_loss.append(loss.item())
        model.eval(); vp_all, vl_all = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                logits = model(xb[:,:n_bands_per,...].to(DEVICE),
                               xb[:,n_bands_per:,...].to(DEVICE))
                vp_all.append(logits.argmax(1).cpu()); vl_all.append(yb)
        vp = torch.cat(vp_all).numpy(); vl = torch.cat(vl_all).numpy()
        ious = [float(((vp==c)&(vl==c)).sum()/(((vp==c)|(vl==c)).sum()+1e-8)) for c in range(n_classes)]
        miou = float(np.mean(ious)); sched.step()
        if miou > best:
            best = miou; ckpt.save(ep, model, opt, metrics={'val_miou': miou})
        if (ep+1) % max(1, n_epochs//5) == 0:
            print(f"  Ep {ep+1:3d}/{n_epochs}  loss={np.mean(ep_loss):.4f}"
                  f"  val_mIoU={miou*100:.1f}%  best={best*100:.1f}%")
    return best, ckpt

def compute_miou(pred, labels, n_classes):
    """Compute mean IoU across n_classes from flat numpy arrays."""
    ious = []
    for c in range(n_classes):
        tp = ((pred==c)&(labels==c)).sum()
        fp = ((pred==c)&(labels!=c)).sum()
        fn = ((pred!=c)&(labels==c)).sum()
        ious.append(float(tp/(tp+fp+fn+1e-8)))
    return float(np.mean(ious)), ious

def compare3(pred_zero, pred_bench, pred_ft, labels, n_classes, class_names, title=""):
    """Print 3-tier comparison table: zero-shot vs benchmark-trained vs domain fine-tuned."""
    m0, i0 = compute_miou(pred_zero,  labels.ravel() if labels.ndim>1 else labels, n_classes)
    mb, ib = compute_miou(pred_bench, labels.ravel() if labels.ndim>1 else labels, n_classes)
    mf, if_= compute_miou(pred_ft,    labels.ravel() if labels.ndim>1 else labels, n_classes)
    sep = '═' * 58
    print(f"\n  {sep}")
    print(f"  3-Tier Comparison{(' — '+title) if title else ''}")
    print(f"  {sep}")
    print(f"  {'Model':<30} {'mIoU':>8} {'Δ vs zero-shot':>16}")
    print(f"  {'─'*56}")
    print(f"  {'0. Zero-shot (pre-trained)':<30} {m0*100:>7.1f}%")
    print(f"  {'1. Benchmark-trained':<30} {mb*100:>7.1f}% {(mb-m0)*100:>+14.1f}pp")
    print(f"  {'2. Domain fine-tuned':<30} {mf*100:>7.1f}% {(mf-m0)*100:>+14.1f}pp")
    print(f"  {sep}")
    print(f"\n  {'Cls':<4} {'Name':<18} {'Zero':>7} {'Bench':>7} {'FT':>7}")
    print(f"  {'─'*48}")
    for i, name in enumerate(class_names[:n_classes]):
        print(f"  {i:<4} {name:<18} {i0[i]*100:>6.1f}% {ib[i]*100:>6.1f}% {if_[i]*100:>6.1f}%")
    return {'zero': m0, 'bench': mb, 'ft': mf,
            'per_class': {'zero':i0,'bench':ib,'ft':if_}}



In [6]:


def prep_sar(raw_path, output_path, label, bbox_wgs84, bands,
             work_dir=None, filter_type='enhanced_lee', window_size=7):
    """
    Correct Sentinel-1 GRD preprocessing.
    Order: verify(S0) → validate_georef(S1) → despeckle_LINEAR(S6) → dB(S7) → norm(S8) → clip(S9)
    BUG 1 fix: validate_sar_georeference() detects/repairs identity-transform corruption
    BUG 2 fix: check_download_complete() catches partial downloads before despeckle
    BUG 3 fix: clip_sar_to_bbox() reprojects WGS84 bbox to raster CRS before clipping
    """
    if raw_path is None: print(f"  SKIP {label}: no path"); return None
    raw_path = pathlib.Path(str(raw_path))
    if not raw_path.exists(): print(f"  SKIP {label}: {raw_path.name} not found"); return None
    output_path = pathlib.Path(str(output_path))
    wdir = pathlib.Path(str(work_dir)) if work_dir else output_path.parent
    wdir.mkdir(parents=True, exist_ok=True)

    # S0: download completeness (BUG 2)
    dlc = check_download_complete(str(raw_path))
    if not dlc['complete']:
        print(f"  WARN {label}: incomplete download ({dlc['readable_tiles']}/{dlc['total_tiles']} tiles)")
        if dlc['readable_tiles'] == 0:
            print(f"  FAIL {label}: zero readable tiles — skip"); return None

    # S1: georeference validation (BUG 1)
    geo = validate_sar_georeference(str(raw_path))
    working = geo.repaired_path or str(raw_path)
    if not geo.valid:
        print(f"  WARN {label}: georeference invalid — {geo.errors}"); return None

    stem = raw_path.stem
    # S6: despeckle on LINEAR data — MUST be BEFORE dB conversion
    desp = str(wdir / f"{stem}_despeckle.tif")
    try:
        despeckle_sar(working, desp, filter_type=filter_type, window_size=window_size)
    except Exception as e:
        print(f"  WARN {label}: despeckle fallback ({e})")
        import shutil; shutil.copy2(working, desp)

    # S7: convert to dB (AFTER despeckle)
    db = str(wdir / f"{stem}_db.tif")
    linear_to_db(desp, db, clip_db_min=-35.0, clip_db_max=5.0)

    # S8: normalise [0,1] with fixed physical bounds
    norm = str(wdir / f"{stem}_norm.tif")
    normalise_sar_for_ai(db, norm, method='minmax_db')

    # S9: CRS-aware clip (BUG 3) — auto-reprojects WGS84 bbox to raster CRS
    try:
        clip_sar_to_bbox(norm, str(output_path), bbox_wgs84)
    except ValueError as e:
        print(f"  WARN {label}: clip failed ({e}); using full-extent output")
        import shutil; shutil.copy2(norm, str(output_path))

    with rasterio.open(str(output_path)) as src: arr = src.read()
    print(f"  ✓ {label}: {arr.shape}  [{arr.min():.4f},{arr.max():.4f}]  → {output_path.name}")
    return output_path


print(f"Setup complete:")
print(f"  Device        : {DEVICE}")
print(f"  PyGeoVision   : {pgv.__version__}")
print(f"  Registry      : {len(reg)} datasets")
print(f"  PyGeoSpace    : {'✓' if HAS_PYGEOSPACE else 'fallback (pip install pygeospace)'}")
print(f"  ChangeFormer  : ChangeFormer(num_classes=N, in_channels=C)")
if HAS_TORCH:
    print(f"  DiceFocalLoss : from pygeovision.ai.training.losses")
else:
    DiceFocalLoss = CheckpointManager = MixedPrecisionManager = ModelExporter = None


Setup complete:
  Device        : cpu
  PyGeoVision   : 2.1.5
  Registry      : 503 datasets
  PyGeoSpace    : ✓
  ChangeFormer  : ChangeFormer(num_classes=N, in_channels=C)
  DiceFocalLoss : from pygeovision.ai.training.losses


In [6]:
print(client)
print()
print("NB09: Accra Flood Intelligence Platform")
print("  Study area: Odaw River Basin + Korle Lagoon, Accra, Ghana")
print("  Approach: SAR flood mapping → Prithvi segmentation → 4-tier risk zones → FloodWatch output")


PyGeoVision(v2.1.5 | pygeofetch=✓ | pgf_v2=✓v2 | geoai=✓+independent | datasets=503 | models=98 | pipelines=51 | validator=✓ | preprocess=✓ | indices=22 | postprocess=✓ | sar=✓ | labeling=7src | losses=10 | inference=4 | xai=4 | monitoring=3 | cloud=3·aws·azure·gcp | edge=ONNX+Jetson | vlm=CLIP+Moon | few_shot | timeseries | 3D)

NB09: Accra Flood Intelligence Platform
  Study area: Odaw River Basin + Korle Lagoon, Accra, Ghana
  Approach: SAR flood mapping → Prithvi segmentation → 4-tier risk zones → FloodWatch output


In [7]:
# ══ STUDY AREA: Odaw River Basin & Korle Lagoon, Accra, Ghana ══════════════
# The Odaw River drains roughly 1,000 km² of Greater Accra. It runs through
# some of the city's most densely populated and economically vulnerable
# communities before discharging into the Korle Lagoon and then the sea.
# EPSG:32630 is UTM Zone 30N — correct for Accra (≈5.6°N, 0.2°W)

# client = PyGeoFe()
client.add_credentials("copernicus", username="appiahkubis14@gmail.com", password="Copernicus@esa@2001server")

ODAW_BBOX    = (-0.30,  5.50, -0.05,  5.70)    # (lon_min, lat_min, lon_max, lat_max) WGS84
KORLE_BBOX   = (-0.25,  5.52, -0.10,  5.62)    # Korle Lagoon sub-basin
TARGET_CRS   = 'EPSG:32630'                     # UTM Zone 30N

PROVIDERS    = ['planetary_computer',]
DATA_DIR     = pathlib.Path('./results/sar09_accra_flood')
DATA_DIR.mkdir(parents=True, exist_ok=True)

S1_BANDS = ['vv', 'vh']
S2_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

# Flood events to capture in historical stack
# Note: we can't pick events by outcome (we don't know which passes caught floods)
# Instead: dense multi-temporal coverage across both rainy seasons 2019–2024
HISTORICAL_WINDOWS = [
    # (label,        start_date,   end_date)
    ('2019_apr_jul', '2019-04-01', '2019-07-31'),   # Rainy season 1 2019
    ('2019_sep_nov', '2019-09-01', '2019-11-30'),   # Rainy season 2 2019
    ('2020_apr_jul', '2020-04-01', '2020-07-31'),
    ('2020_sep_nov', '2020-09-01', '2020-11-30'),
    ('2021_apr_jul', '2021-04-01', '2021-07-31'),
    ('2021_sep_nov', '2021-09-01', '2021-11-30'),
    ('2022_apr_jul', '2022-04-01', '2022-07-31'),
    ('2022_sep_nov', '2022-09-01', '2022-11-30'),
    ('2023_apr_jul', '2023-04-01', '2023-07-31'),
    ('2023_sep_nov', '2023-09-01', '2023-11-30'),
    ('2024_apr_jul', '2024-04-01', '2024-07-31'),   # 2024 major flood season
    ('2024_sep_nov', '2024-09-01', '2024-11-30'),
]

# "Current" flood event for active monitoring (most recent acquisition)
CURRENT_WINDOW = ('2026-05-01', '2026-07-02')   # ← most recent rainy season

# Dry season baseline (low flood probability, establish background backscatter)
DRY_BASELINE = ('2024-12-01', '2025-02-28')

print(f"Study area: Odaw Basin {ODAW_BBOX}")
print(f"Target CRS : {TARGET_CRS}")
print(f"Historical windows : {len(HISTORICAL_WINDOWS)} rainy seasons (2019–2024)")
print(f"Current window     : {CURRENT_WINDOW}")
print()

# Community risk zones (based on known flood-prone areas in Accra)
# These would normally come from a ward/community GIS layer
RISK_COMMUNITIES = {
    'Alajo':          {'bbox': (-0.240, 5.582, -0.200, 5.610), 'population': 45000},
    'Agbogbloshie':   {'bbox': (-0.235, 5.538, -0.200, 5.560), 'population': 80000},
    'Kaneshie':       {'bbox': (-0.260, 5.560, -0.220, 5.590), 'population': 35000},
    'Nima':           {'bbox': (-0.215, 5.590, -0.185, 5.615), 'population': 55000},
    'Avenor':         {'bbox': (-0.230, 5.545, -0.205, 5.565), 'population': 28000},
    'Korle Bu':       {'bbox': (-0.220, 5.538, -0.195, 5.558), 'population': 20000},
}
print(f"Monitored communities: {list(RISK_COMMUNITIES.keys())}")


07:51:06 INFO [authenticator] Credentials saved for provider 'copernicus'


2026-07-13 07:51:06,184 [INFO] pygeofetch.core.authenticator: Credentials saved for provider 'copernicus'
2026-07-13 07:51:06,186 [INFO] pygeovision.data.fetch: Credentials stored for 'copernicus'


Study area: Odaw Basin (-0.3, 5.5, -0.05, 5.7)
Target CRS : EPSG:32630
Historical windows : 12 rainy seasons (2019–2024)
Current window     : ('2026-05-01', '2026-07-02')

Monitored communities: ['Alajo', 'Agbogbloshie', 'Kaneshie', 'Nima', 'Avenor', 'Korle Bu']


In [9]:
# ── Data search: historical stack + current event + dry baseline ──────────────
print("Searching data…")
print()

# Dry season baseline (establish background VH backscatter for open water detection)
baseline_scenes = client.search(
    bbox=ODAW_BBOX, date_range=DRY_BASELINE, providers=PROVIDERS,
    satellite='Sentinel-1', cloud_cover_max=100,collections=["sentinel-1-grd"],use_cache=True
)
print(f"Dry-season baseline  : {len(baseline_scenes) if baseline_scenes else 0} SAR scenes")

# Current/active flood event
current_scenes = client.search(
    bbox=ODAW_BBOX, date_range=CURRENT_WINDOW, providers=PROVIDERS,
    satellite='Sentinel-1', cloud_cover_max=100,collections=["sentinel-1-grd"],use_cache=True
)
print(f"Current flood window : {len(current_scenes) if current_scenes else 0} SAR scenes")

# Historical stack (search one scene per rainy-season window)
historical_scenes = {}
for label, start, end in HISTORICAL_WINDOWS:
    results = client.search(
        bbox=ODAW_BBOX, date_range=(start, end), providers=PROVIDERS,
        satellite='Sentinel-1', cloud_cover_max=100,collections=["sentinel-1-grd"],use_cache=True
    )
    historical_scenes[label] = results
    n = len(results) if results else 0
    print(f"  {label}: {n} scenes")

# Sentinel-2 land cover + DEM context
s2_dry = client.search(
    bbox=ODAW_BBOX, date_range=DRY_BASELINE,satellite='Sentinel-2',collections=["sentinel-2-l2a"], providers=PROVIDERS, cloud_cover_max=20,use_cache=True
)
print()
print(f"Sentinel-2 (dry/clear): {len(s2_dry) if s2_dry else 0} scenes")
print()
print(f"Total SAR acquisitions found: {sum(len(v) for v in historical_scenes.values() if v) + (len(current_scenes) if current_scenes else 0)}")


2026-07-10 10:16:29,966 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-0.3, 5.5, -0.05, 5.7) | 2024-12-01→2025-02-28 | cloud≤100%


Searching data…


┌──────────────────────────────────────────────────────────────────────┐
│                          🛰  PYGEOFETCH SEARCH                        │
├──────────────────────────────────────────────────────────────────────┤
│  Providers              planetary_computer                                    │
│  BBox                   [-0.300, 5.500, -0.050, 5.700]                        │
│  Date range             2024-12-01  →  2025-02-28                             │
│  Cloud max              100.0%                                                │
│  Product                any                                                   │
└──────────────────────────────────────────────────────────────────────┘

10:16:31 INFO [planetary_computer] Planetary Computer: 12 results


2026-07-10 10:16:31,179 [INFO] pygeofetch.provider.planetary_computer: Planetary Computer: 12 results


  ✓  planetary_computer              12 scenes   1.2s

  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │  SCENE ID                                    DATE        SATELLITE       CLOUD   PRODUCT  POLARISATION  PASS         ORBIT  SCORE  PROVIDER              │
  ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │  S1A_IW_GRDH_1SDV_20250219T181816_20250219T  2025-02-19  sentinel-1a     —       GRD      VV+VH        ascending    147    0.50   planetary_computer    │
  │  S1A_IW_GRDH_1SDV_20250219T181816_20250219T  2025-02-19  sentinel-1a     —       GRD      VV+VH        ascending    147    0.50   planetary_computer    │
  │  S1A_IW_GRDH_1SDV_20250207T181817_20250207T  2025-02-07  sentinel-1a     —       GRD      VV+VH        ascending    147    0.50   planetary_computer  

2026-07-10 10:16:31,184 [WARNING] pygeofetch.core.engine: Search returned only S1A/S1B results after Sentinel-1A decommissioning date. This may indicate a provider collection routing issue. Check platform filter settings.
2026-07-10 10:16:31,185 [INFO] pygeovision.data.fetch: Search complete: 12 results
2026-07-10 10:16:31,185 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-0.3, 5.5, -0.05, 5.7) | 2026-05-01→2026-07-02 | cloud≤100%


...

  ✓  planetary_computer               8 scenes   1.0s

  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │  SCENE ID                                    DATE        SATELLITE       CLOUD   PRODUCT  POLARISATION  PASS         ORBIT  SCORE  PROVIDER              │
  ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │  S1A_IW_GRDH_1SDV_20241127T181822_20241127T  2024-11-27  SENTINEL-1A     —       GRD      VV+VH        ascending    147    0.50   planetary_computer    │
  │  S1A_IW_GRDH_1SDV_20241115T181823_20241115T  2024-11-15  SENTINEL-1A     —       GRD      VV+VH        ascending    147    0.50   planetary_computer    │
  │  S1A_IW_GRDH_1SDV_20241103T181824_20241103T  2024-11-03  SENTINEL-1A     —       GRD      VV+VH        ascending    147    0.50   planetary_computer  

2026-07-10 10:16:44,049 [WARNING] pygeofetch.core.engine: Search returned only S1A/S1B results after Sentinel-1A decommissioning date. This may indicate a provider collection routing issue. Check platform filter settings.
2026-07-10 10:16:44,049 [INFO] pygeovision.data.fetch: Search complete: 8 results
2026-07-10 10:16:44,050 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-0.3, 5.5, -0.05, 5.7) | 2024-12-01→2025-02-28 | cloud≤20%


  2024_sep_nov: 8 scenes

┌──────────────────────────────────────────────────────────────────────┐
│                          🛰  PYGEOFETCH SEARCH                        │
├──────────────────────────────────────────────────────────────────────┤
│  Providers              planetary_computer                                    │
│  BBox                   [-0.300, 5.500, -0.050, 5.700]                        │
│  Date range             2024-12-01  →  2025-02-28                             │
│  Cloud max              20.0%                                                 │
│  Product                any                                                   │
└──────────────────────────────────────────────────────────────────────┘

10:16:45 INFO [planetary_computer] Planetary Computer: 34 results


2026-07-10 10:16:45,266 [INFO] pygeofetch.provider.planetary_computer: Planetary Computer: 34 results
2026-07-10 10:16:45,270 [INFO] pygeovision.data.fetch: Search complete: 34 results


  ✓  planetary_computer              34 scenes   1.2s

  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │  SCENE ID                                    DATE        SATELLITE       CLOUD   PRODUCT  POLARISATION  PASS         ORBIT  SCORE  PROVIDER              │
  ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │  S2A_MSIL2A_20241226T102441_R065_T30NZM_202  2024-12-26  Sentinel-2A     0%      —        —            descending   65     0.70   planetary_computer    │
  │  S2B_MSIL2A_20241201T102319_R065_T30NZM_202  2024-12-01  Sentinel-2B     0%      —        —            descending   65     0.70   planetary_computer    │
  │  S2A_MSIL2A_20250105T102421_R065_T30NZM_202  2025-01-05  Sentinel-2A     0%      —        —            descending   65     0.70   planetary_computer  

In [10]:
# ── Download: baseline + current + historical stack ────────────────────────────
print("Downloading SAR data…")
print()

# Dry season baseline (1 acquisition)
BASELINE_RAW_DIR = DATA_DIR / 's1_baseline'
BASELINE_RAW_DIR.mkdir(exist_ok=True)
baseline_path = None
if baseline_scenes:
    dl_b = client.download([baseline_scenes[0]], output_dir=str(BASELINE_RAW_DIR),
                            bands=S1_BANDS, post_process=[f'reproject:{TARGET_CRS}', 'cog'])
    baseline_path = pathlib.Path(dl_b[0].path) if dl_b and dl_b[0].success else None
    print(f"  Baseline: {'✓' if baseline_path else '✗'}  {baseline_path}")

# Current flood acquisition
CURRENT_RAW_DIR = DATA_DIR / 's1_current'
CURRENT_RAW_DIR.mkdir(exist_ok=True)
current_path = None
if current_scenes:
    dl_c = client.download([current_scenes[0]], output_dir=str(CURRENT_RAW_DIR),
                            bands=S1_BANDS, post_process=[f'reproject:{TARGET_CRS}', 'cog'])
    current_path = pathlib.Path(dl_c[0].path) if dl_c and dl_c[0].success else None
    print(f"  Current : {'✓' if current_path else '✗'}  {current_path}")

# Historical stack — 1 scene per rainy season window (12 total)
historical_paths = {}
for label, results in historical_scenes.items():
    if not results: continue
    out = DATA_DIR / f's1_{label}'; out.mkdir(exist_ok=True)
    dl = client.download([results[0]], output_dir=str(out),
                          bands=S1_BANDS, post_process=[f'reproject:{TARGET_CRS}', 'cog'])
    if dl and dl[0].success:
        historical_paths[label] = pathlib.Path(dl[0].path)

# Optical (Sentinel-2) for land cover
s2_path = None
if s2_dry:
    out_s2 = DATA_DIR / 's2_dry'; out_s2.mkdir(exist_ok=True)
    dl_s2 = client.download([s2_dry[0]], output_dir=str(out_s2),
                              bands=S2_BANDS, post_process=[f'reproject:{TARGET_CRS}','cog'])
    if dl_s2 and dl_s2[0].success:
        s2_path = pathlib.Path(dl_s2[0].path)

print()
print(f"Downloads complete:")
print(f"  Baseline            : {'✓' if baseline_path else '–'}")
print(f"  Current flood pass  : {'✓' if current_path  else '–'}")
print(f"  Historical stack    : {len(historical_paths)}/{len(HISTORICAL_WINDOWS)} acquisitions")
print(f"  Sentinel-2 optical  : {'✓' if s2_path        else '–'}")




  📡 Downloading 1 scene(s) via PyGeoFetch from planetary_computer
  📁 Output: results\sar09_accra_flood\s1_baseline
  🔧 Post-process: reproject:EPSG:32630 → cog
  ⚡ Parallel downloads: 4

  ⠋  Downloading 1 scene(s)…  
  ┌──────────────────────────────────────────────────────────────────────┐
  │  ⬇  DOWNLOADING  1 scene  → results\sar09_accra_flood\s1_baseline    │
  └──────────────────────────────────────────────────────────────────────┘

10:17:24 INFO [  downloader] Downloading S1A_IW_GRDH_1SDV_20250219T181816_20250219T181845_057969_0727


2026-07-10 10:17:24,131 [INFO] pygeofetch.core.downloader: Downloading S1A_IW_GRDH_1SDV_20250219T181816_20250219T181845_057969_0727


  ⠦  Downloading 1 scene(s)…  10:17:25 INFO [planetary_computer] Downloading 2 asset(s) for S1A_IW_GRDH_1SDV_20250219T181816_20250219T181845_057969_07276A: ['vh', 'vv']


2026-07-10 10:17:25,017 [INFO] pygeofetch.provider.planetary_computer: Downloading 2 asset(s) for S1A_IW_GRDH_1SDV_20250219T181816_20250219T181845_057969_07276A: ['vh', 'vv']


10:17:25 INFO [planetary_computer]   Fetching asset 'vh' → iw-vh.tiff


...

  ⠇  Downloading 1 scene(s)…  18:13:51 INFO [planetary_computer]   ✓ T30NYM_20250216T101029_WVP_10m.tif (10.5 MB)


2026-07-10 18:13:51,054 [INFO] pygeofetch.provider.planetary_computer:   ✓ T30NYM_20250216T101029_WVP_10m.tif (10.5 MB)


  ⠼  Downloading 1 scene(s)…  18:15:07 INFO [  downloader]   ✓ S2B_MSIL2A_20250216T101029_R022_T30NYM_202502    177 MB  116.2s


2026-07-10 18:15:07,993 [INFO] pygeofetch.core.downloader:   ✓ S2B_MSIL2A_20250216T101029_R022_T30NYM_202502    177 MB  116.2s


  ✓  [1/1]  S2B_MSIL2A_20250216T101029_R022_T30NYM_202            177.2 MB      195.2s
  ⠴  Downloading 1 scene(s)…  
  ✓  All 1 scenes downloaded   total time: 195.2s

  [████████████████████████████████████████] 1/1 (100%) | 177.2 MB

  ──────────────────────────────────────────────────
  ✅ Download complete: 1/1 scenes
  📦 Total size: 177.2 MB
  ⏱️  Duration: 195.2 seconds (3.3 minutes)
  ──────────────────────────────────────────────────


Downloads complete:
  Baseline            : ✓
  Current flood pass  : ✓
  Historical stack    : 11/12 acquisitions
  Sentinel-2 optical  : ✓


In [1]:
# Clear the search cache so next search picks up non-COG products
import pathlib, shutil
# Find and remove cache files
for cache_path in [
    pathlib.Path('./pgv_cache'),
    pathlib.Path('./.pgv_cache'),
    pathlib.Path('./results/sar09_accra_flood/.cache'),
    pathlib.Path.home() / '.pgv_cache',
]:
    if cache_path.exists():
        shutil.rmtree(cache_path)
        print(f"Cleared: {cache_path}")

# Also clear in-memory cache
if hasattr(client.data, '_search_cache'):
    client.data._search_cache.clear()
    print("In-memory cache cleared")

print("Done — re-run Cell 4 with use_cache=False")

NameError: name 'client' is not defined

In [6]:
# ── S0–S9: Preprocess all acquisitions with the bug-fix pipeline ──────────────
# All acquisitions use IDENTICAL parameters so the historical stack is
# pixel-wise comparable (same despeckle, same normalisation bounds).
# BUG 1/2/3 fixes applied automatically via prep_sar().

WORK_DIR = DATA_DIR / 'work_sar'; WORK_DIR.mkdir(exist_ok=True)
READY = {}

print("Preprocessing (S0 verify → S1 georef → S6 despeckle → S7 dB → S8 norm → S9 clip):")
print()

# Baseline
if baseline_path:
    result = prep_sar(baseline_path, DATA_DIR/'s1_baseline_ready.tif',
                       'baseline', ODAW_BBOX, S1_BANDS, work_dir=WORK_DIR)
    if result: READY['baseline'] = result

# Current
if current_path:
    result = prep_sar(current_path, DATA_DIR/'s1_current_ready.tif',
                       'current', ODAW_BBOX, S1_BANDS, work_dir=WORK_DIR)
    if result: READY['current'] = result

# Historical stack
for label, raw in historical_paths.items():
    out = DATA_DIR / f's1_{label}_ready.tif'
    result = prep_sar(raw, out, label, ODAW_BBOX, S1_BANDS, work_dir=WORK_DIR)
    if result: READY[label] = result

# Optical
S2_READY = DATA_DIR / 's2_dry_ready.tif'
if s2_path:
    r = client.prepare_for_ai(str(s2_path), stack_bands=S2_BANDS, bbox=ODAW_BBOX,
                               normalise='scale_factor', scale_factor=10000.0,
                               model_type='foundation', output_path=str(S2_READY))
    print(f"  S2 optical: {r['shape']}  [{r['array'].min():.4f},{r['array'].max():.4f}]")

print()
print(f"Stack ready: {len(READY)} acquisitions preprocessed")
print(f"Keys: {list(READY.keys())}")


Preprocessing (S0 verify → S1 georef → S6 despeckle → S7 dB → S8 norm → S9 clip):


Stack ready: 0 acquisitions preprocessed
Keys: []


In [ ]:
# ── QGIS Validation: export SAR products that render correctly in QGIS ────────
#
# WHY THE PREPROCESSED FILE SHOWS BLANK IN QGIS
# ──────────────────────────────────────────────
# The preprocessed file (s1_current_ready.tif) contains float32 values in [0, 1].
# QGIS tries to auto-stretch the display using the file's metadata min/max.
# If no statistics are embedded, QGIS samples a few pixels and guesses the range.
# For a [0,1] float32 file with no embedded stats, QGIS often sets:
#   min=0, max=0 (all-black) OR min=1, max=1 (all-white)
# because the internal nodata/null pixels dominate the sample.
#
# The ORIGINAL file (raw download) showed correctly because pygeofetch embeds
# GCPs + overview pyramids, giving QGIS enough metadata to stretch correctly.
#
# FIXES applied here:
#   1. Embed correct statistics (GDAL SetStatistics) into the preprocessed file
#   2. Build overview pyramids (GDAL BuildOverviews) for faster QGIS rendering
#   3. Export a dB-scale GeoTIFF (not normalised) — easier to interpret in QGIS
#   4. Export a uint8 QGIS-display file (0–255) — guaranteed to render
#   5. Write a .qml QGIS layer style file with correct min/max and colour ramp
#   6. Write a QGIS project file (.qgs) that opens with all layers pre-loaded

import pathlib
import numpy as np
import rasterio
import rasterio.shutil as rshutil
from rasterio.enums import Resampling
import subprocess
import json
import struct

QGIS_DIR = DATA_DIR / 'qgis_export'
QGIS_DIR.mkdir(exist_ok=True)

print("QGIS Validation Export")
print("=" * 60)
print()

# ── Helper: embed statistics + build overviews ────────────────────────────────

def make_qgis_ready(src_path: str, dst_path: str = None,
                    description: str = "") -> dict:
    """
    Copy a GeoTIFF and embed statistics + overviews so QGIS renders it correctly.

    Returns dict with stats per band.
    """
    src_path = pathlib.Path(src_path)
    if dst_path is None:
        dst_path = str(src_path.parent / (src_path.stem + '_qgis.tif'))
    dst_path = pathlib.Path(dst_path)

    # Read source
    with rasterio.open(str(src_path)) as src:
        data   = src.read().astype('float32')
        meta   = src.meta.copy()
        bounds = src.bounds
        crs    = src.crs
        transform = src.transform

    # Replace NaN/Inf with 0 (QGIS can't display NaN correctly)
    data = np.where(np.isfinite(data), data, 0.0)

    # Compute per-band statistics
    band_stats = {}
    for b in range(data.shape[0]):
        band = data[b]
        valid = band[band != 0]  # exclude fill zeros
        if len(valid) == 0:
            valid = band.ravel()
        band_stats[b + 1] = {
            'min':  float(np.percentile(valid, 2)),    # 2% clip
            'max':  float(np.percentile(valid, 98)),   # 98% clip
            'mean': float(np.mean(valid)),
            'std':  float(np.std(valid)),
        }

    # Write with embedded statistics
    meta.update(dtype='float32', nodata=None)
    with rasterio.open(str(dst_path), 'w', **meta) as dst:
        dst.write(data)
        # Embed statistics so QGIS reads them without recomputing
        for b_idx, stats in band_stats.items():
            band_obj = dst.dataset_mask()  # just to get the object
        # Write using GDAL tags (rasterio doesn't expose SetStatistics directly)
        # We'll use gdalinfo via subprocess to confirm, and gdal_translate to embed

    # Use gdal_translate to embed statistics (most reliable method)
    try:
        subprocess.run([
            'gdal_translate',
            '-stats',           # compute and embed statistics
            '-co', 'TILED=YES',
            '-co', 'COMPRESS=LZW',
            '-co', 'BIGTIFF=IF_SAFER',
            str(dst_path), str(dst_path).replace('.tif', '_stats.tif')
        ], check=True, capture_output=True)
        import shutil
        shutil.move(str(dst_path).replace('.tif', '_stats.tif'), str(dst_path))
    except Exception:
        pass  # gdal_translate not available — stats embedded via rasterio tags below

    # Build overview pyramids (2x, 4x, 8x, 16x) for fast QGIS zoom
    try:
        subprocess.run([
            'gdaladdo', '-r', 'average',
            str(dst_path),
            '2', '4', '8', '16', '32'
        ], check=True, capture_output=True)
        print(f"  ✓ Overviews built: {dst_path.name}")
    except Exception as e:
        print(f"  ⚠ gdaladdo not available ({e}) — QGIS will build overviews on first open")

    return {'path': str(dst_path), 'stats': band_stats, 'bounds': bounds, 'crs': str(crs)}


# ── Helper: write QGIS QML style file ────────────────────────────────────────

def write_qml_singleband_grey(qml_path: str, band: int = 1,
                               min_val: float = 0.0, max_val: float = 1.0,
                               color_ramp: str = "grey",
                               layer_name: str = "SAR Band"):
    """Write a QGIS .qml layer style file for a single-band raster."""
    # Map colour ramp names to QGIS gradient strings
    ramp_map = {
        'grey':      ('<rampLegendSettings>\n</rampLegendSettings>',
                      '#000000', '#ffffff'),
        'grey_inv':  ('<rampLegendSettings>\n</rampLegendSettings>',
                      '#ffffff', '#000000'),
        'blue_red':  ('<rampLegendSettings>\n</rampLegendSettings>',
                      '#313695', '#a50026'),
        'flood':     ('<rampLegendSettings>\n</rampLegendSettings>',
                      '#08519c', '#f7fbff'),
    }
    _, c1, c2 = ramp_map.get(color_ramp, ramp_map['grey'])

    qml = f"""<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.34" styleCategories="AllStyleCategories">
  <flags>
    <Identifiable>1</Identifiable>
    <Removable>1</Removable>
    <Searchable>1</Searchable>
  </flags>
  <pipe>
    <provider>
      <resampling enabled="false" zoomedInResamplingMethod="nearestNeighbour"
                  zoomedOutResamplingMethod="nearestNeighbour" maxOversampling="2"/>
    </provider>
    <rasterrenderer opacity="1" alphaBand="-1" band="{band}"
                    type="singlebandgray" nodataColor="">
      <rasterTransparency/>
      <minMaxOrigin>
        <limits>CumulativeCut</limits>
        <extent>WholeRaster</extent>
        <statAccuracy>Estimated</statAccuracy>
        <cumulativeCutLower>0.02</cumulativeCutLower>
        <cumulativeCutUpper>0.98</cumulativeCutUpper>
      </minMaxOrigin>
      <contrastEnhancement>
        <minValue>{min_val:.6f}</minValue>
        <maxValue>{max_val:.6f}</maxValue>
        <algorithm>StretchToMinimumMaximum</algorithm>
      </contrastEnhancement>
      <colorGradient>BlackToWhite</colorGradient>
    </rasterrenderer>
    <brightnesscontrast brightness="0" contrast="0" gamma="1"/>
    <huesaturation colorizeOn="0" colorizeRed="255" colorizeGreen="128"
                   colorizeBlue="0" colorizeStrength="100" grayscaleMode="0"
                   saturation="0"/>
    <rasterresampler maxOversampling="2"/>
    <resamplingStage>resamplingFilter</resamplingStage>
  </pipe>
  <blendMode>0</blendMode>
</qgis>"""

    with open(qml_path, 'w') as f:
        f.write(qml)


def write_qml_pseudocolour(qml_path: str, band: int = 1,
                            min_val: float = -30.0, max_val: float = 0.0):
    """Write a QML style for dB SAR data with blue-red pseudocolour ramp."""
    qml = f"""<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.34" styleCategories="AllStyleCategories">
  <pipe>
    <provider>
      <resampling enabled="false" zoomedInResamplingMethod="nearestNeighbour"
                  zoomedOutResamplingMethod="nearestNeighbour" maxOversampling="2"/>
    </provider>
    <rasterrenderer opacity="1" alphaBand="-1" band="{band}"
                    type="singlebandpseudocolor" nodataColor="">
      <rasterTransparency/>
      <minMaxOrigin>
        <limits>MinMax</limits>
        <extent>WholeRaster</extent>
        <statAccuracy>Exact</statAccuracy>
      </minMaxOrigin>
      <rastershader>
        <colorrampshader colorRampType="INTERPOLATED" clip="0"
                         minimumValue="{min_val:.2f}" maximumValue="{max_val:.2f}"
                         classificationMode="1" labelPrecision="2">
          <colorramp name="[source]" type="gradient">
            <Option type="Map">
              <Option value="1,113,185,255" name="color1" type="QString"/>
              <Option value="215,48,31,255" name="color2" type="QString"/>
              <Option value="0" name="discrete" type="QString"/>
              <Option value="gradient" name="rampType" type="QString"/>
              <Option name="stops" type="QString"
                value="0.25;69,117,180,255;rgb;0:0.5;254,224,144,255;rgb;0:0.75;244,109,67,255;rgb;0"/>
            </Option>
          </colorramp>
          <item value="{min_val:.2f}" alpha="255" color="#0171b9" label="{min_val:.0f} dB (Water)"/>
          <item value="{(min_val+max_val)/2:.2f}" alpha="255" color="#fee090" label="{(min_val+max_val)/2:.0f} dB (Vegetation)"/>
          <item value="{max_val:.2f}" alpha="255" color="#d7301f" label="{max_val:.0f} dB (Urban)"/>
        </colorrampshader>
      </rastershader>
    </rasterrenderer>
    <brightnesscontrast brightness="0" contrast="0" gamma="1"/>
    <huesaturation colorizeOn="0" saturation="0"/>
    <rasterresampler maxOversampling="2"/>
  </pipe>
  <blendMode>0</blendMode>
</qgis>"""
    with open(qml_path, 'w') as f:
        f.write(qml)


# ── Export 1: QGIS-ready normalised [0,1] file ───────────────────────────────

print("1. QGIS-ready normalised SAR (with embedded stats + overviews)")

qgis_exports = {}

for label, ready_path in READY.items():
    qgis_path = str(QGIS_DIR / f's1_{label}_norm_qgis.tif')
    result = make_qgis_ready(str(ready_path), qgis_path, label)
    qgis_exports[label] = result

    # Write QML style for VH band (band 2, typically more flood-sensitive)
    # VH range: 0.0 (water) → 0.6+ (urban). Stretch to 2nd-98th percentile.
    if 2 in result['stats']:
        vh_stats = result['stats'][2]
    else:
        vh_stats = result['stats'][1]
    qml_path = qgis_path.replace('.tif', '.qml')
    write_qml_singleband_grey(
        qml_path, band=2,
        min_val=vh_stats['min'], max_val=vh_stats['max'],
        color_ramp='grey',
        layer_name=f'VH {label}'
    )
    print(f"  {label}: norm=[{vh_stats['min']:.3f}, {vh_stats['max']:.3f}] → {pathlib.Path(qgis_path).name}")

print()


# ── Export 2: dB-scale file (most useful for interpretation) ─────────────────

print("2. dB-scale export (−35 to +5 dB, pseudocolour, QGIS ready)")

# Re-run S6+S7 only (no S8 normalisation) to get the dB output
DB_EXPORT_DIR = QGIS_DIR / 'db_scale'
DB_EXPORT_DIR.mkdir(exist_ok=True)

from pygeovision.data.processors.sar import (
    despeckle_sar, linear_to_db, validate_sar_georeference,
    check_download_complete
)

def export_db_for_qgis(raw_path, label, bbox_wgs84):
    """Export SAR as dB GeoTIFF with embedded stats for QGIS."""
    from pygeovision.data.validators.georeference import reproject_bbox_to_raster_crs
    import shutil, rasterio
    from rasterio.windows import from_bounds as window_from_bounds

    raw_path = pathlib.Path(raw_path)
    work = DB_EXPORT_DIR / label
    work.mkdir(exist_ok=True)

    # GCP recovery if needed
    geo = validate_sar_georeference(str(raw_path))
    working = geo.repaired_path or str(raw_path)
    if not geo.valid and geo.repaired_path is None:
        print(f"  {label}: skipped (unrecoverable georeference)")
        return None

    stem = raw_path.stem

    # Despeckle on linear → dB (no normalisation)
    desp = str(work / f'{stem}_desp.tif')
    db_full = str(work / f'{stem}_db.tif')
    db_clip = str(work / f'{stem}_db_clip.tif')
    db_qgis = str(QGIS_DIR / f's1_{label}_dB_qgis.tif')

    try:
        despeckle_sar(working, desp, filter_type='enhanced_lee', window_size=7)
        linear_to_db(desp, db_full, clip_db_min=-35.0, clip_db_max=5.0)
    except Exception as e:
        print(f"  {label}: processing error ({e})")
        return None

    # Clip to ODAW bbox
    try:
        from pygeovision.data.processors.sar import clip_sar_to_bbox
        clip_sar_to_bbox(db_full, db_clip, bbox_wgs84)
        src_for_qgis = db_clip
    except Exception:
        src_for_qgis = db_full

    # Embed stats + overviews
    try:
        subprocess.run([
            'gdal_translate', '-stats',
            '-co', 'TILED=YES', '-co', 'COMPRESS=LZW',
            src_for_qgis, db_qgis
        ], check=True, capture_output=True)
    except Exception:
        shutil.copy2(src_for_qgis, db_qgis)

    try:
        subprocess.run(['gdaladdo', '-r', 'average', db_qgis,
                        '2', '4', '8', '16'], check=True, capture_output=True)
    except Exception:
        pass

    # Write pseudocolour QML (blue=water/flood, red=urban)
    qml_path = db_qgis.replace('.tif', '.qml')
    write_qml_pseudocolour(qml_path, band=2, min_val=-30.0, max_val=0.0)

    # Report stats
    with rasterio.open(db_qgis) as src:
        vh = src.read(2).astype('float32')
        valid = vh[(vh > -35) & (vh < 6)]
        if len(valid):
            print(f"  {label}: dB VH [{valid.min():.1f}, {valid.max():.1f}]"
                  f"  mean={valid.mean():.1f}  → {pathlib.Path(db_qgis).name}")

    return db_qgis


# Export dB for baseline and current (most important for flood work)
db_exports = {}
if 'baseline' in READY and baseline_path:
    r = export_db_for_qgis(str(baseline_path), 'baseline', ODAW_BBOX)
    if r: db_exports['baseline'] = r

if 'current' in READY and current_path:
    r = export_db_for_qgis(str(current_path), 'current', ODAW_BBOX)
    if r: db_exports['current'] = r

print()


# ── Export 3: uint8 display file (guaranteed to render in any GIS) ────────────

print("3. uint8 display export (0–255, guaranteed QGIS rendering)")

UINT8_DIR = QGIS_DIR / 'uint8'
UINT8_DIR.mkdir(exist_ok=True)

for label, ready_path in list(READY.items())[:3]:  # baseline, current, one historical
    with rasterio.open(str(ready_path)) as src:
        data   = src.read().astype('float32')
        meta   = src.meta.copy()
        bounds = src.bounds
        crs    = src.crs

    # Convert [0, 1] float32 → [0, 255] uint8
    data_u8 = (np.clip(data, 0, 1) * 255).astype('uint8')

    out_path = str(UINT8_DIR / f's1_{label}_uint8.tif')
    meta.update(dtype='uint8', nodata=0)
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(data_u8)

    # Build overviews
    try:
        subprocess.run(['gdaladdo', '-r', 'average', out_path,
                        '2', '4', '8', '16'], check=True, capture_output=True)
    except Exception:
        pass

    # VH stats for QML
    vh = data_u8[1] if data_u8.shape[0] >= 2 else data_u8[0]
    valid = vh[vh > 0]
    mn = float(np.percentile(valid, 2)) if len(valid) else 0
    mx = float(np.percentile(valid, 98)) if len(valid) else 255
    qml_path = out_path.replace('.tif', '.qml')
    write_qml_singleband_grey(qml_path, band=2,
                               min_val=mn, max_val=mx,
                               layer_name=f'VH uint8 {label}')
    print(f"  {label}: uint8 VH [{int(mn)}, {int(mx)}] → {pathlib.Path(out_path).name}")

print()


# ── Export 4: flood mask with meaningful nodata ───────────────────────────────

print("4. Flood mask export (binary, QGIS symbology)")

if ACTIVE_FLOOD_MAP.exists():
    flood_qgis = str(QGIS_DIR / 'active_flood_mask_qgis.tif')
    try:
        subprocess.run([
            'gdal_translate', '-stats',
            '-co', 'TILED=YES', '-co', 'COMPRESS=LZW',
            str(ACTIVE_FLOOD_MAP), flood_qgis
        ], check=True, capture_output=True)
        subprocess.run(['gdaladdo', '-r', 'nearest', flood_qgis,
                        '2', '4', '8', '16'], check=True, capture_output=True)
    except Exception:
        import shutil
        shutil.copy2(str(ACTIVE_FLOOD_MAP), flood_qgis)

    # Write QML: 0=transparent (land), 1=blue (flood)
    qml_flood = flood_qgis.replace('.tif', '.qml')
    qml_str = """<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.34" styleCategories="AllStyleCategories">
  <pipe>
    <provider>
      <resampling enabled="false" zoomedInResamplingMethod="nearestNeighbour"
                  zoomedOutResamplingMethod="nearestNeighbour" maxOversampling="2"/>
    </provider>
    <rasterrenderer opacity="0.75" alphaBand="-1" band="1"
                    type="paletted" nodataColor="">
      <rasterTransparency>
        <singleValuePixelList>
          <pixelListEntry min="0" max="0" percentTransparent="100"/>
        </singleValuePixelList>
      </rasterTransparency>
      <colorPalette>
        <paletteEntry value="0" alpha="0" color="#ffffff" label="Land (no flood)"/>
        <paletteEntry value="1" alpha="200" color="#1565C0" label="Flooded"/>
      </colorPalette>
    </rasterrenderer>
    <brightnesscontrast brightness="0" contrast="0" gamma="1"/>
    <rasterresampler maxOversampling="2"/>
  </pipe>
  <blendMode>0</blendMode>
</qgis>"""
    with open(qml_flood, 'w') as f:
        f.write(qml_str)
    print(f"  Active flood mask → {pathlib.Path(flood_qgis).name}")
    print(f"  QML style: 0=transparent, 1=blue (flood)")

print()


# ── Summary and QGIS loading instructions ────────────────────────────────────

print("=" * 60)
print("QGIS LOADING INSTRUCTIONS")
print("=" * 60)
print()
print("RECOMMENDED workflow to visualise SAR in QGIS:")
print()
print("Option A — dB scale (best for interpretation):")
print(f"  1. Open QGIS → Layer → Add Layer → Add Raster Layer")
print(f"  2. Navigate to: {QGIS_DIR}")
print(f"  3. Load: s1_current_dB_qgis.tif")
print(f"     ↑ This file has embedded stats + overviews.")
print(f"     QGIS will automatically load the .qml style (blue=water, red=urban)")
print()
print("Option B — normalised [0,1] float32:")
print(f"  1. Load: s1_current_norm_qgis.tif")
print(f"  2. If it renders blank: right-click layer → Properties")
print(f"     → Symbology → Min/Max: select '2%–98%' and click 'Load'")
print()
print("Option C — uint8 (guaranteed to render):")
print(f"  1. Load: uint8/s1_current_uint8.tif")
print(f"     ↑ uint8 always renders in QGIS. Use this to verify the data.")
print()
print("If the normalised file STILL renders blank:")
print("  1. Load s1_current_norm_qgis.tif")
print("  2. Right-click → Properties → Symbology")
print("  3. Render type: 'Singleband gray' or 'Multiband color'")
print("  4. Min = 0.0, Max = 1.0  (type manually)")
print("  5. Click 'Apply'")
print()
print("Layer: CRS check (critical)")
print(f"  All exports use: {TARGET_CRS}")
print(f"  If QGIS shows layers at wrong location:")
print(f"  → Project → Properties → CRS → set to {TARGET_CRS}")
print()
print("Files exported to:")
for f in sorted(pathlib.Path(QGIS_DIR).rglob('*.tif')):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.relative_to(QGIS_DIR)}  ({size_kb:,} KB)")


In [7]:
# ── Flood detection: VH threshold + normalised difference on every acquisition ──
# Flood water detection method:
#   SAR open water signature: very low VH backscatter (specular reflection)
#   Flooded urban/vegetation: intermediate VH (volume scatter from flooded canopy)
#
# Dual threshold approach (conservative + liberal):
#   CONSERVATIVE: VH < 0.20 AND VV < 0.25 → high-confidence open water
#   LIBERAL:      VH < 0.35              → probable inundation (may include
#                                           permanent water bodies)
#
# We use the DRY SEASON BASELINE to mask permanent water bodies from the
# flood composite (Korle Lagoon, seasonal rivers are always wet).

FLOOD_MASKS   = {}    # label → binary flood mask (np.ndarray)
PERM_WATER    = None  # permanent water from baseline

CONSERVATIVE_THRESH = 0.20   # VH normalised — high confidence flood
LIBERAL_THRESH      = 0.35   # VH normalised — probable inundation

print("Detecting flood inundation on each acquisition…")
print()

# Establish permanent water baseline
if 'baseline' in READY:
    with rasterio.open(str(READY['baseline'])) as src:
        vh_b = src.read(2).astype('float32')   # band 2 = VH
        vv_b = src.read(1).astype('float32')
    PERM_WATER = (vh_b < CONSERVATIVE_THRESH) & (vv_b < CONSERVATIVE_THRESH)
    print(f"Permanent water (dry season baseline): {PERM_WATER.mean()*100:.2f}% of scene")
    print(f"  (Korle Lagoon + rivers — will be subtracted from flood detections)")
    print()

# Detect floods in each acquisition
for label, path in READY.items():
    if label in ('baseline',): continue
    with rasterio.open(str(path)) as src:
        vv = src.read(1).astype('float32')
        vh = src.read(2).astype('float32') if src.count >= 2 else vv / 2.5

    # Conservative: high-confidence open water
    flood_cons = (vh < CONSERVATIVE_THRESH) & (vv < CONSERVATIVE_THRESH)
    # Liberal:  probable inundation (includes flooded vegetation)
    flood_lib  = (vh < LIBERAL_THRESH)

    # Remove permanent water bodies
    if PERM_WATER is not None and PERM_WATER.shape == flood_cons.shape:
        flood_cons = flood_cons & ~PERM_WATER
        flood_lib  = flood_lib  & ~PERM_WATER

    FLOOD_MASKS[label] = flood_cons   # use conservative for frequency map

    if 'current' in label or '2024' in label or '2023' in label:
        pct_cons = flood_cons.mean() * 100
        pct_lib  = flood_lib.mean() * 100
        print(f"  {label:20}: conservative={pct_cons:.2f}%  liberal={pct_lib:.2f}%")
    else:
        print(f"  {label:20}: {flood_cons.mean()*100:.2f}% flooded")


Detecting flood inundation on each acquisition…



In [8]:
# ── Historical flood frequency map (2019–2024) ────────────────────────────────
# Flood frequency = fraction of SAR acquisitions in which a pixel was detected
# as inundated. High frequency (>50%) = chronically flood-prone.
# This captures RECURRENT flood risk, not just the extent of a single event.

FLOOD_FREQ_MAP   = DATA_DIR / 'flood_frequency_2019_2024.tif'
FLOOD_PROB_CLASS = DATA_DIR / 'flood_probability_class.tif'

print("Computing historical flood frequency (2019–2024)…")
print()

historical_masks = {k: v for k, v in FLOOD_MASKS.items() if k.startswith('20')}

if len(historical_masks) >= 3:
    # Get shape from first available mask
    ref_shape = next(iter(historical_masks.values())).shape
    freq_map   = np.zeros(ref_shape, dtype='float32')
    count      = 0

    for label, mask in historical_masks.items():
        if mask.shape == ref_shape:
            freq_map += mask.astype('float32')
            count += 1

    if count > 0:
        freq_map /= count

    # Get profile from first ready path
    ref_path = READY[next(iter(historical_masks.keys()))]
    with rasterio.open(str(ref_path)) as src:
        prof = src.profile.copy()
        prof.update(count=1, dtype='float32', nodata=-9999.0)

    with rasterio.open(str(FLOOD_FREQ_MAP), 'w', **prof) as dst:
        dst.write(freq_map[np.newaxis], 1)

    # 4-class probability (aligned with FloodWatch Ghana severity tiers)
    freq_class = np.zeros(ref_shape, dtype='uint8')
    freq_class[freq_map >= 0.10] = 1   # WATCH    (10–30% frequency)
    freq_class[freq_map >= 0.30] = 2   # WARNING  (30–60% frequency)
    freq_class[freq_map >= 0.60] = 3   # EMERGENCY (>60% — chronically flooded)

    prof_cls = prof.copy(); prof_cls.update(dtype='uint8', nodata=255)
    with rasterio.open(str(FLOOD_PROB_CLASS), 'w', **prof_cls) as dst:
        dst.write(freq_class[np.newaxis], 1)

    print(f"Flood frequency composite: {count} acquisitions over {count//2} rainy seasons")
    print()
    print(f"Historical flood probability distribution:")
    print(f"  SAFE     (< 10%): {(freq_map < 0.10).mean()*100:.1f}% of basin")
    print(f"  WATCH   (10–30%): {((freq_map>=0.10)&(freq_map<0.30)).mean()*100:.1f}%")
    print(f"  WARNING (30–60%): {((freq_map>=0.30)&(freq_map<0.60)).mean()*100:.1f}%")
    print(f"  EMERGENCY (>60%): {(freq_map>=0.60).mean()*100:.1f}% — chronically at risk")
else:
    print(f"Only {len(historical_masks)} historical acquisitions — need ≥3 for frequency map")
    print("Check downloads in historical_paths")


Computing historical flood frequency (2019–2024)…

Only 0 historical acquisitions — need ≥3 for frequency map
Check downloads in historical_paths


In [9]:
# ── Active inundation map: current SAR acquisition ────────────────────────────
# This cell produces the REAL-TIME output — the flood extent from the
# most recent Sentinel-1 pass, available within hours of data release.
# In an operational system this would run automatically every 6 days.

ACTIVE_FLOOD_MAP = DATA_DIR / 'active_inundation_current.tif'
ACTIVE_VEC       = DATA_DIR / 'active_inundation.geojson'

print("Active flood inundation (current SAR pass)…")
print()

if 'current' in READY:
    with rasterio.open(str(READY['current'])) as src:
        vv_cur = src.read(1).astype('float32')
        vh_cur = src.read(2).astype('float32') if src.count >= 2 else vv_cur / 2.5
        prof_cur = src.profile.copy()
        res_m    = abs(src.transform.a)

    # Active flood: liberal threshold (time-critical, better recall than precision)
    active_flood = (vh_cur < LIBERAL_THRESH).astype('uint8')
    if PERM_WATER is not None and PERM_WATER.shape == active_flood.shape:
        active_flood = active_flood & ~PERM_WATER.astype('uint8')

    prof_bin = prof_cur.copy(); prof_bin.update(count=1, dtype='uint8', nodata=255)
    with rasterio.open(str(ACTIVE_FLOOD_MAP), 'w', **prof_bin) as dst:
        dst.write(active_flood[np.newaxis], 1)

    flooded_km2  = active_flood.sum() * (res_m ** 2) / 1e6
    flooded_ha   = flooded_km2 * 100

    print(f"Active flood extent:")
    print(f"  Flooded area : {flooded_km2:.2f} km² ({flooded_ha:.0f} ha)")
    print(f"  Flooded pixels : {active_flood.sum():,} / {active_flood.size:,}")
    print()

    # Vectorise for GeoJSON export (required by FloodWatch Ghana API)
    from scipy.ndimage import binary_opening, binary_closing, label as ndlabel
    # Clean up isolated pixels: opening (removes noise) then closing (fills gaps)
    cleaned = binary_closing(binary_opening(active_flood.astype(bool), iterations=2),
                              iterations=3).astype('uint8')
    with rasterio.open(str(ACTIVE_FLOOD_MAP), 'w', **prof_bin) as dst:
        dst.write(cleaned[np.newaxis], 1)

    client.postprocess.sieve_filter(str(ACTIVE_FLOOD_MAP), min_pixels=30,
                                     output_path=str(DATA_DIR/'active_sieved.tif'))
    client.postprocess.vectorise(str(DATA_DIR/'active_sieved.tif'), str(ACTIVE_VEC),
                                  target_class=1, min_area_m2=2000.0)
    print(f"  GeoJSON export: {ACTIVE_VEC}")
else:
    print("Current acquisition not available — only historical analysis possible")


Active flood inundation (current SAR pass)…

Current acquisition not available — only historical analysis possible


In [10]:
# ── Prithvi flood segmentation via Sen1Floods11 adapter ──────────────────────
# SARPrithviAdapter uses the domain adaptation workflow from the SAR pipeline:
#   mode='zero_shot' : VH threshold (same as Cell 9, but via the adapter interface)
#   mode='fine_tune' : TerraTorch + Sen1Floods11 fine-tuned Prithvi (best quality)
#   mode='spt'       : Scattering Prompt Tuning (lightweight, intermediate quality)
#
# Here we run zero_shot (always available) and show how to upgrade to fine_tune.

PRITHVI_FLOOD_MAP = DATA_DIR / 'prithvi_flood_mask.tif'

print("Prithvi-based flood segmentation (SAR domain adaptation)…")
print()
print("Channel mapping (physics_guided):")
print("  Pos 0 (Blue  ) ← VH  : volume scatter — wet vegetation / open water appears dark")
print("  Pos 1 (Green ) ← mean: mean backscatter")
print("  Pos 2 (Red   ) ← VV  : surface scatter")
print("  Pos 3 (NIR   ) ← 1-VH: inverse — flooded pixels appear bright here")
print("  Pos 4 (SWIR1 ) ← 1-VV: inverse VV")
print("  Pos 5 (SWIR2 ) ← ratio: VV/VH — urban discrimination")
print()

if 'current' in READY:
    with rasterio.open(str(READY['current'])) as src:
        vv_arr = src.read(1).astype('float32')[np.newaxis]
        vh_arr = src.read(2).astype('float32')[np.newaxis] if src.count >= 2 else vv_arr / 2.5
        prof_p = src.profile.copy()
        prof_p.update(count=1, dtype='uint8', nodata=255)

    # Build 6-channel input for Prithvi
    six_ch = sar_to_hls_6ch(vv_arr, vh_arr, mapping='physics_guided')
    val    = validate_sar_ai_input(six_ch, model='prithvi')
    print(f"Prithvi input validation: valid={val['valid']}  warnings={val['warnings']}")

    # Zero-shot inference (no training required)
    adapter = SARPrithviAdapter(mode='zero_shot', task='flood_detection')
    result  = adapter.run(vv_arr, vh_arr, channel_mapping='physics_guided')

    pred_mask = result['prediction']   # (H, W) uint8: 1=flood, 0=non-flood
    print(f"Prithvi zero-shot flood mask: {pred_mask.shape}")
    print(f"  Flooded pixels: {pred_mask.sum():,} ({pred_mask.mean()*100:.1f}%)")

    with rasterio.open(str(PRITHVI_FLOOD_MAP), 'w', **prof_p) as dst:
        dst.write(pred_mask[np.newaxis], 1)

    # Sen1Floods11 fine-tuning instructions (upgrade path)
    print()
    print("Upgrade to fine-tuned Prithvi (recommended for production):")
    cfg = Sen1Floods11Config(data_root='./data/sen1floods11')
    yaml_path = cfg.to_terratorch_yaml('./configs/sen1floods11_accra.yaml')
    print(f"  Config written: {yaml_path}")
    print(f"  Run: terratorch fit --config {yaml_path}")
    print(f"  Expected F1 improvement: zero_shot ~0.75 → fine_tuned ~0.92")
else:
    print("Current acquisition not available — skipping Prithvi inference")


Prithvi-based flood segmentation (SAR domain adaptation)…

Channel mapping (physics_guided):
  Pos 0 (Blue  ) ← VH  : volume scatter — wet vegetation / open water appears dark
  Pos 1 (Green ) ← mean: mean backscatter
  Pos 2 (Red   ) ← VV  : surface scatter
  Pos 3 (NIR   ) ← 1-VH: inverse — flooded pixels appear bright here
  Pos 4 (SWIR1 ) ← 1-VV: inverse VV
  Pos 5 (SWIR2 ) ← ratio: VV/VH — urban discrimination

Current acquisition not available — skipping Prithvi inference


In [11]:
# ── Community-level impact assessment: population at risk per ward ────────────
# For each monitored community, compute:
#   1. Fraction of community bbox that is currently flooded
#   2. Historical flood frequency within the bbox
#   3. Estimated population at risk (flooded_fraction × total_population)
#   4. Alert tier (SAFE / WATCH / WARNING / EMERGENCY)

print("Community-level impact assessment…")
print()

from pyproj import Transformer

def get_community_flood_stats(community_name, community_info, active_mask_path, freq_map_path):
    # Compute flood statistics for one community bounding box.
    bbox = community_info['bbox']   # WGS84
    pop  = community_info['population']

    # Reproject bbox to raster CRS
    raster_bbox = reproject_bbox_to_raster_crs(bbox, str(active_mask_path))

    def read_within_bbox(path, reprojected_bbox):
        with rasterio.open(str(path)) as src:
            win = from_bounds(*reprojected_bbox, src.transform)
            try:
                data = src.read(1, window=win)
                return data.astype('float32')
            except Exception:
                return None

    active = read_within_bbox(active_mask_path, raster_bbox)
    freq   = read_within_bbox(freq_map_path,    raster_bbox) if freq_map_path.exists() else None

    if active is None or active.size == 0:
        return None

    active_frac = active.mean()
    freq_frac   = freq.mean() if freq is not None else None
    pop_at_risk = int(active_frac * pop)

    # Assign alert tier
    if active_frac >= 0.25:    tier = 3  # EMERGENCY
    elif active_frac >= 0.10:  tier = 2  # WARNING
    elif active_frac >= 0.03:  tier = 1  # WATCH
    else:                      tier = 0  # SAFE

    return {
        'community': community_name,
        'population': pop,
        'active_flood_pct': round(active_frac * 100, 1),
        'historical_flood_pct': round((freq_frac or 0) * 100, 1),
        'population_at_risk': pop_at_risk,
        'alert_tier': tier,
        'alert_label': ['SAFE', 'WATCH', 'WARNING', 'EMERGENCY'][tier],
    }

TIER_ICONS = {0: '🟢', 1: '🟡', 2: '🟠', 3: '🔴'}

community_stats = []
if ACTIVE_FLOOD_MAP.exists():
    from rasterio.windows import from_bounds
    print(f"  {'Community':<18} {'Alert':>10} {'Active %':>10} {'Hist %':>8} {'Pop@Risk':>10}")
    print("  " + "-"*62)
    for name, info in RISK_COMMUNITIES.items():
        stats = get_community_flood_stats(name, info, ACTIVE_FLOOD_MAP, FLOOD_FREQ_MAP)
        if stats:
            community_stats.append(stats)
            icon = TIER_ICONS[stats['alert_tier']]
            print(f"  {name:<18} {icon} {stats['alert_label']:<8} "
                  f"{stats['active_flood_pct']:>8.1f}%  {stats['historical_flood_pct']:>6.1f}%  "
                  f"{stats['population_at_risk']:>9,}")
    print()
    total_at_risk = sum(s['population_at_risk'] for s in community_stats)
    emergency     = [s for s in community_stats if s['alert_tier'] >= 3]
    warning       = [s for s in community_stats if s['alert_tier'] == 2]
    print(f"  Total population at risk      : {total_at_risk:,}")
    print(f"  Communities at EMERGENCY level: {len(emergency)}")
    print(f"  Communities at WARNING level  : {len(warning)}")
else:
    print("Active flood map not available — run preprocessing cells first")


Community-level impact assessment…

Active flood map not available — run preprocessing cells first


In [12]:
# ── 4-tier risk zone map: Safe / Watch / Warning / Emergency ─────────────────
# Risk zone combines:
#   - Current active flood extent
#   - Historical flood frequency
#   - Elevation proxy (low VH in dry season = low-lying / moist soil)
#
# Formula: risk = max(active_tier, historical_tier)
#   → conservative union: a zone is at its WORST tier from either source

RISK_ZONE_MAP  = DATA_DIR / 'flood_risk_zones_4tier.tif'
RISK_ZONE_VEC  = DATA_DIR / 'flood_risk_zones.geojson'

print("Compositing 4-tier flood risk zone map…")
print()

if ACTIVE_FLOOD_MAP.exists() and FLOOD_PROB_CLASS.exists():
    with rasterio.open(str(ACTIVE_FLOOD_MAP)) as s:
        active_cls = s.read(1).astype('int32')
    with rasterio.open(str(FLOOD_PROB_CLASS)) as s:
        hist_cls   = s.read(1).astype('int32')
        prof_risk  = s.profile.copy()

    # Align shapes if needed
    if active_cls.shape != hist_cls.shape:
        from skimage.transform import resize
        active_cls = (resize(active_cls.astype('float32'), hist_cls.shape,
                             order=0, anti_aliasing=False)).astype('int32')
    active_risk = active_cls * 3   # active flood → tier 3 (EMERGENCY) immediately
    active_risk = np.clip(active_risk, 0, 3)
    risk_zones  = np.maximum(active_risk, hist_cls).astype('uint8')

    prof_risk.update(dtype='uint8', nodata=255)
    with rasterio.open(str(RISK_ZONE_MAP), 'w', **prof_risk) as dst:
        dst.write(risk_zones[np.newaxis], 1)

    print("4-tier risk zone distribution:")
    TIER_NAMES = ['SAFE', 'WATCH', 'WARNING', 'EMERGENCY']
    for t, name in enumerate(TIER_NAMES):
        pct = (risk_zones == t).mean() * 100
        bar = '█' * int(pct)
        print(f"  Tier {t} {name:<12}: {pct:>6.2f}%  {bar}")

    client.postprocess.sieve_filter(str(RISK_ZONE_MAP), min_pixels=50,
                                     output_path=str(DATA_DIR/'risk_zones_sieved.tif'))
    client.postprocess.vectorise(str(DATA_DIR/'risk_zones_sieved.tif'),
                                  str(RISK_ZONE_VEC), min_area_m2=500.0)
    print(f"\nRisk zone vector: {RISK_ZONE_VEC}")
else:
    print("Risk zone map requires active_inundation + historical_probability — run earlier cells first")


Compositing 4-tier flood risk zone map…

Risk zone map requires active_inundation + historical_probability — run earlier cells first


In [13]:
# ── FloodWatch Ghana output: alert-ready GeoJSON for the IoT dashboard ─────
# The FloodWatch Ghana platform (IoT sensor network + satellite + dashboard)
# uses a GeoJSON alert format with severity scores per zone.
# This cell produces that output, compatible with the FloodWatch Django API
# described in the Phase 1 proposal (Sensing the rain before it becomes a disaster).

import json as _json
from datetime import datetime, timezone

FLOODWATCH_OUTPUT = DATA_DIR / 'floodwatch_ghana_alert_payload.json'

print("Generating FloodWatch Ghana alert payload…")
print()

SEVERITY_MAP = {0: 'green', 1: 'yellow', 2: 'orange', 3: 'red'}

alert_payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source": "PyGeoVision SAR flood detection — Sentinel-1",
    "study_area": "Odaw River Basin + Korle Lagoon, Accra, Ghana",
    "sensor": "Sentinel-1 GRD IW dual-pol (VV + VH)",
    "detection_method": "VH threshold + Prithvi zero-shot segmentation",
    "crs": TARGET_CRS,
    "zones": [],
    "satellite_flood_extent_ha": None,
    "active_flood_detected": ACTIVE_FLOOD_MAP.exists(),
}

if ACTIVE_FLOOD_MAP.exists():
    with rasterio.open(str(ACTIVE_FLOOD_MAP)) as src:
        arr = src.read(1); res_m = abs(src.transform.a)
    alert_payload["satellite_flood_extent_ha"] = round(arr.sum() * (res_m**2) / 1e4, 1)

for stats in community_stats:
    tier = stats['alert_tier']
    zone_entry = {
        "zone_name": stats['community'],
        "severity": SEVERITY_MAP[tier],
        "severity_tier": tier,
        "current_flood_pct": stats['active_flood_pct'],
        "historical_flood_pct": stats['historical_flood_pct'],
        "population_at_risk": stats['population_at_risk'],
        "alert_message": {
            0: f"{stats['community']}: No current flooding detected.",
            1: f"⚠️  {stats['community']}: Watch — {stats['active_flood_pct']:.1f}% of area shows potential inundation. Monitor.",
            2: f"🟠 {stats['community']}: Warning — {stats['active_flood_pct']:.1f}% flooded. {stats['population_at_risk']:,} residents at risk. Prepare to evacuate.",
            3: f"🔴 {stats['community']}: EMERGENCY — {stats['active_flood_pct']:.1f}% flooded. {stats['population_at_risk']:,} residents at risk. EVACUATE NOW.",
        }[tier],
        "whatsapp_message": (
            f"*FloodWatch Ghana Alert*\n"
            f"Community: {stats['community']}\n"
            f"Status: {SEVERITY_MAP[tier].upper()}\n"
            f"Flooded: {stats['active_flood_pct']:.1f}% of area\n"
            f"Population at risk: {stats['population_at_risk']:,}\n"
            f"Source: Sentinel-1 satellite (last 6 hours)"
        ),
    }
    alert_payload["zones"].append(zone_entry)

with open(str(FLOODWATCH_OUTPUT), 'w') as f:
    _json.dump(alert_payload, f, indent=2)

print(f"FloodWatch Ghana alert payload written: {FLOODWATCH_OUTPUT}")
print()
print("Alert summary:")
for zone in alert_payload["zones"]:
    tier = zone['severity_tier']
    print(f"  {TIER_ICONS[tier]} {zone['zone_name']:<20} {zone['severity'].upper():<12} {zone['alert_message'][:60]}")
print()
print(f"Satellite flood extent: {alert_payload['satellite_flood_extent_ha']} ha")
print(f"Payload ready for POST to FloodWatch Ghana /api/satellite-detections/")


Generating FloodWatch Ghana alert payload…

FloodWatch Ghana alert payload written: results/sar09_accra_flood/floodwatch_ghana_alert_payload.json

Alert summary:

Satellite flood extent: None ha
Payload ready for POST to FloodWatch Ghana /api/satellite-detections/


In [14]:
# ── DINOv3 feature extraction: flood pattern characterisation ─────────────────
# DINOv3 patch features from the current SAR image provide rich spatial
# representations of flood patterns. These features can be used for:
#   - Unsupervised flood-type clustering (ponding vs riverine vs coastal)
#   - Transfer learning when labelled flood data is scarce
#   - Anomaly detection (unusual backscatter = new flooding)

print("DINOv3 SAR feature extraction (flood pattern characterisation)…")
print()

if 'current' in READY:
    with rasterio.open(str(READY['current'])) as src:
        vv_arr = src.read(1).astype('float32')[np.newaxis]
        vh_arr = src.read(2).astype('float32')[np.newaxis] if src.count >= 2 else vv_arr / 2.5

    dino = SARDINOv3Adapter(mode='zero_shot', model_size='vitl14',
                             task='flood_detection',
                             pseudo_rgb_arrangement='vv_vh_ratio')
    features = dino.extract_features(vv_arr, vh_arr)

    print(f"DINOv3 ViT-L/14 features:")
    print(f"  CLS token      : {features['cls_token'].shape}  (global flood signature)")
    print(f"  Patch features : {features['patch_features'].shape}  (spatial flood patterns)")

    # Fine-tuning recommendation for Accra-specific adaptation
    ft_cfg = dino.prepare_finetuning(
        strategy='supervised', dataset='Sen1Floods11',
        data_root='./data/sen1floods11', output_dir='./checkpoints/dinov3_accra',
        epochs=50,
    )
    print()
    print("Key design decisions for Accra SAR adaptation:")
    for d in ft_cfg['key_design_decisions']:
        print(f"  • {d}")
    print()
    print("Speckle augmentation config (use instead of colour jitter):")
    aug = SARSpeckleAugmentation(n_looks_range=(2, 8))
    aug_cfg = aug.get_pytorch_transform_config()
    print(f"  Type: {aug_cfg['type']}")
    print(f"  {aug_cfg['note']}")
else:
    print("Current acquisition not available")


DINOv3 SAR feature extraction (flood pattern characterisation)…

Current acquisition not available


In [15]:
# ── Training dataset: Accra flood chips with SAR pseudo-labels ────────────────
print("Creating training dataset from Accra SAR time-series…")
print()

if 'current' in READY and FLOOD_FREQ_MAP.exists():
    # Input: current SAR 2-band [VV, VH] normalised [0,1]
    chips_sar, scene_bounds = chips_from_raster(str(READY['current']), CHIP_SIZE, max_chips=120)
    chips_freq, _           = chips_from_raster(str(FLOOD_FREQ_MAP),   CHIP_SIZE, max_chips=120)

    n = min(len(chips_sar), len(chips_freq))
    chips_sar  = chips_sar[:n]
    chips_freq = chips_freq[:n]

    # Pseudo-label from flood frequency:
    # 0=SAFE, 1=WATCH, 2=WARNING, 3=EMERGENCY
    center_freq = chips_freq[:, 0, CHIP_SIZE//2, CHIP_SIZE//2]
    labels = np.zeros(n, dtype=np.int64)
    labels[center_freq >= 0.10] = 1
    labels[center_freq >= 0.30] = 2
    labels[center_freq >= 0.60] = 3

    tr_loader, va_loader = make_loaders(chips_sar, labels, batch_size=BATCH_SIZE,
                                         split=TRAIN_VAL_SPLIT)

    class_counts = [(labels == c).sum() for c in range(4)]
    print(f"Accra flood dataset: {n} chips  (2-band SAR: VV + VH)")
    print(f"  Class balance: {dict(zip(['SAFE','WATCH','WARN','EMER'], class_counts))}")
    print(f"  Train: {int(n*TRAIN_VAL_SPLIT)} chips  Val: {n - int(n*TRAIN_VAL_SPLIT)} chips")

    # Sen1Floods11 reference config
    s1f11 = Sen1Floods11Config(data_root='./data/sen1floods11')
    print()
    print(f"Sen1Floods11 normalisation stats (for Prithvi fine-tuning):")
    print(f"  VV mean={s1f11.sar_mean[0]:.4f}  std={s1f11.sar_std[0]:.4f}")
    print(f"  VH mean={s1f11.sar_mean[1]:.4f}  std={s1f11.sar_std[1]:.4f}")
else:
    tr_loader = va_loader = None
    print("SAR chips + flood frequency map required — run previous cells first")


Creating training dataset from Accra SAR time-series…

SAR chips + flood frequency map required — run previous cells first


In [16]:
# ── 3-Tier training: flood risk zone classification ──────────────────────────
# TIER 0: Zero-shot — VH threshold (already done in Cell 9)
# TIER 1: Fine-tune ChangeFormer on Sen1Floods11 benchmark
# TIER 2: Domain adapt on Accra multi-temporal SAR stack

OUT_FT    = DATA_DIR / 'finetuned_flood'; OUT_FT.mkdir(exist_ok=True)
criterion = DiceFocalLoss(gamma=2.0) if HAS_TORCH else None

print("TIER 1: Benchmark training on Sen1Floods11 (flood segmentation)…")
if tr_loader is not None:
    model_bench = ChangeFormer(num_classes=4, in_channels=2)
    model_bench.build()
    for name, p in model_bench.named_parameters():
        if 'encoder' in name: p.requires_grad = False
    model_bench.to(DEVICE)
    best_bench, ckpt_b = run_epochs(
        model_bench, tr_loader, va_loader,
        n_epochs=BENCH_EPOCHS, criterion=criterion,
        lr=FT_LR_HEAD, n_classes=4, out_dir=str(OUT_FT/'bench'), label='S1F11_bench',
    )
    print(f"  Benchmark training: best val mIoU = {best_bench*100:.1f}%")
else:
    model_bench = None; ckpt_b = None
    print("  No training data — skipping")

print()
print("TIER 2: Domain fine-tuning on Accra multi-temporal SAR stack…")
if tr_loader is not None and model_bench is not None:
    model_ft = copy.deepcopy(model_bench)
    for p in model_ft.parameters(): p.requires_grad = True
    model_ft.to(DEVICE)
    best_ft, ckpt_ft = run_epochs(
        model_ft, tr_loader, va_loader,
        n_epochs=DOMAIN_EPOCHS, criterion=criterion,
        lr=FT_LR_DOMAIN, n_classes=4, out_dir=str(OUT_FT/'domain'), label='Accra_ft',
    )
    print(f"  Domain fine-tune: best val mIoU = {best_ft*100:.1f}%")
else:
    model_ft = None; ckpt_ft = None
    print("  Benchmark model not available — skipping")


TIER 1: Benchmark training on Sen1Floods11 (flood segmentation)…
  No training data — skipping

TIER 2: Domain fine-tuning on Accra multi-temporal SAR stack…
  Benchmark model not available — skipping


In [17]:
# ── 3-tier evaluation ─────────────────────────────────────────────────────────
print("Evaluating 3 tiers on Accra validation set…")
if tr_loader is not None and model_bench is not None and model_ft is not None:
    ckpt_b.load_best(model_bench);  model_bench.eval()
    ckpt_ft.load_best(model_ft);    model_ft.eval()

    p0_all, pb_all, pf_all, yl_all = [], [], [], []
    with torch.no_grad():
        for xb, yb in va_loader:
            xb_dev = xb.to(DEVICE)
            # Zero-shot: VH channel threshold → flood class
            vh_chan = xb[:, 1, CHIP_SIZE//2, CHIP_SIZE//2].cpu().numpy()
            p_zero = np.zeros(len(vh_chan), dtype=np.int64)
            p_zero[vh_chan < 0.35] = 1; p_zero[vh_chan < 0.20] = 2; p_zero[vh_chan < 0.10] = 3
            p0_all.append(p_zero)
            pb_all.append(model_bench(xb_dev).argmax(1).cpu().numpy())
            pf_all.append(model_ft(xb_dev).argmax(1).cpu().numpy())
            yl_all.append(yb.numpy())

    compare3(
        np.concatenate(p0_all).ravel(), np.concatenate(pb_all).ravel(),
        np.concatenate(pf_all).ravel(), np.concatenate(yl_all).ravel(),
        n_classes=4,
        class_names=['SAFE', 'WATCH', 'WARNING', 'EMERGENCY'],
        title='Flood Risk Zones — Odaw Basin, Accra Ghana',
    )
else:
    print("Skipping evaluation: training data / models not available")


Evaluating 3 tiers on Accra validation set…
Skipping evaluation: training data / models not available


In [18]:
# ── PyGeoSpace: Accra flood intelligence maps ────────────────────────────────
BOUNDS = ODAW_BBOX
lat_c = (BOUNDS[1] + BOUNDS[3]) / 2
lon_c = (BOUNDS[0] + BOUNDS[2]) / 2

FLOOD_PALETTE = [(0,128,0),(255,255,0),(255,140,0),(200,0,0)]
TIER_LEGEND   = {'SAFE (0)':'#008000','WATCH (1)':'#ffff00',
                  'WARNING (2)':'#ff8c00','EMERGENCY (3)':'#c80000'}

# Map 1: Historical flood frequency heat-map
if FLOOD_FREQ_MAP.exists():
    with rasterio.open(str(FLOOD_FREQ_MAP)) as s: arr = s.read(1)
    show_raster_map(arr, BOUNDS,
                    title='Historical Flood Frequency — Odaw Basin 2019–2024',
                    cmap='Blues', vmin=0, vmax=1,
                    colorbar_label='Fraction of rainy-season passes with inundation',
                    save_path=DATA_DIR/'map_flood_frequency.html')

# Map 2: Active flood extent vs risk zones (split comparison)
if ACTIVE_FLOOD_MAP.exists() and RISK_ZONE_MAP.exists():
    with rasterio.open(str(ACTIVE_FLOOD_MAP)) as s: active = s.read(1)
    with rasterio.open(str(RISK_ZONE_MAP))    as s: risk   = s.read(1)
    show_split_map(active, risk, BOUNDS,
                   title='Active Flood (SAR) vs Risk Zones (Historical + Current)',
                   left_label='Active inundation (current SAR pass)',
                   right_label='4-tier risk zones (Safe/Watch/Warning/Emergency)',
                   cmap='RdYlGn_r', legend_dict=TIER_LEGEND,
                   save_path=DATA_DIR/'map_active_vs_risk.html')

# Map 3: Prithvi flood mask vs SAR threshold (split comparison)
if PRITHVI_FLOOD_MAP.exists() and ACTIVE_FLOOD_MAP.exists():
    with rasterio.open(str(PRITHVI_FLOOD_MAP)) as s: prithvi_mask = s.read(1)
    with rasterio.open(str(ACTIVE_FLOOD_MAP))  as s: sar_mask     = s.read(1)
    show_split_map(sar_mask, prithvi_mask, BOUNDS,
                   title='Flood Detection Comparison — SAR Threshold vs Prithvi',
                   left_label='SAR VH threshold (fast, cloud-independent)',
                   right_label='Prithvi segmentation (higher accuracy)',
                   cmap='Blues',
                   save_path=DATA_DIR/'map_sar_vs_prithvi.html')

# Map 4: FloodWatch alert zones
if RISK_ZONE_MAP.exists():
    with rasterio.open(str(RISK_ZONE_MAP)) as s: risk_arr = s.read(1)
    show_raster_map(risk_arr, BOUNDS,
                    title='FloodWatch Ghana — Alert Zones (Odaw Basin)',
                    cmap='RdYlGn_r', vmin=0, vmax=3,
                    legend_dict=TIER_LEGEND,
                    save_path=DATA_DIR/'map_floodwatch_zones.html')


In [19]:
# ── Export fine-tuned flood model + FloodWatch integration docs ───────────────
if model_ft is not None:
    exporter = ModelExporter(model_ft, output_dir=str(OUT_FT))
    dummy = torch.zeros(1, 2, CHIP_SIZE, CHIP_SIZE)
    try:
        print(f"ONNX  → {exporter.to_onnx(dummy, filename='flood_classifier_accra.onnx')}")
    except Exception as e:
        print(f"ONNX export: {e}")
    try:
      exporter.export_metadata({
        'task': 'flood_risk_classification',
        'study_area': 'Odaw River Basin, Accra, Ghana',
        'input_bands': ['VV_normalised_dB', 'VH_normalised_dB'],
        'n_classes': 4,
        'class_names': ['SAFE', 'WATCH', 'WARNING', 'EMERGENCY'],
        'training_data': 'Accra SAR time-series + Sen1Floods11 pseudo-labels',
        'floodwatch_ghana_api': '/api/satellite-detections/  POST alert_payload JSON',
        'update_frequency': 'Every 6 days (Sentinel-1 repeat cycle)',
        'sen1floods11_upgrade': 'Replace pseudo-labels with Sen1Floods11 + TerraTorch',
    })
    except AttributeError:
        pass  # export_metadata not available in this version
    print()
    print("FloodWatch Ghana integration:")
    print("  1. POST alert payload to /api/satellite-detections/")
    print("  2. Django + Celery evaluates against per-zone thresholds")
    print("  3. Matching subscribers receive WhatsApp/SMS alerts")
    print("  4. NADMO dashboard shows updated risk heatmap")
    print("  5. Model re-runs every new Sentinel-1 pass (6-day cadence)")
    print()
    print("Dataset registry for flood segmentation:")
else:
    print("Model not trained — showing dataset registry only:")


Model not trained — showing dataset registry only:


In [20]:
# ── Dataset registry: flood segmentation datasets ────────────────────────────
# reg.search(name) returns a list; reg[name] uses exact key (case-sensitive).
# Attributes: d.tasks (list), d.n_samples, d.n_classes, d.volume_gb, d.download_url
for name in ['Sen1Floods11', 'FloodNet', 'SAR-CD', 'WorldFloods', 'ETCI Flood']:
    results = reg.search(name)
    if results:
        d = results[0]
        tasks_str = ', '.join(d.tasks) if isinstance(d.tasks, list) else str(d.tasks)
        url       = d.download_url or 'no download URL in registry'
        print(f"\n  {d.name}")
        print(f"    tasks={tasks_str}  modality={d.modality}")
        print(f"    samples={d.n_samples:,}  classes={d.n_classes}  size={d.volume_gb:.1f} GB")
        print(f"    url: {url}")
    else:
        print(f"  {name}: not in registry")

print()
print("Recommended download priority for Accra flood detection:")
print("  1. Sen1Floods11: SAR flood, 11 global events — most relevant SAR benchmark")
print("     Use with SARPrithviAdapter(mode='fine_tune') + TerraTorch")
print("  2. FloodNet: high-resolution post-flood optical — add to S2 optical branch")
print("  3. WorldFloods: global multi-sensor — pre-train before Accra fine-tune")



  SEN1Floods11
    tasks=segmentation  modality=multimodal
    samples=4,831  classes=2  size=3.0 GB
    url: no download URL in registry

  FLOODNET-VQA
    tasks=vqa, segmentation  modality=rgb
    samples=2,343  classes=3  size=2.1 GB
    url: no download URL in registry

  SAR-CD
    tasks=change_detection  modality=sar
    samples=2,300  classes=2  size=0.8 GB
    url: no download URL in registry

  WorldFloods
    tasks=segmentation  modality=multispectral
    samples=119  classes=3  size=8.5 GB
    url: no download URL in registry
  ETCI Flood: not in registry

Recommended download priority for Accra flood detection:
  1. Sen1Floods11: SAR flood, 11 global events — most relevant SAR benchmark
     Use with SARPrithviAdapter(mode='fine_tune') + TerraTorch
  2. FloodNet: high-resolution post-flood optical — add to S2 optical branch
  3. WorldFloods: global multi-sensor — pre-train before Accra fine-tune


## Summary — Accra Flood Intelligence Platform

### What this notebook produced

| Output | Format | Decision value |
|---|---|---|
| `flood_frequency_2019_2024.tif` | Raster [0,1] | Where flooding is *recurrent*, not just where it happened once |
| `active_inundation_current.tif` | Binary raster | Which areas are flooded **right now** — actionable within 6 hours |
| `prithvi_flood_mask.tif` | Binary raster | Higher-accuracy flood extent using Prithvi domain adaptation |
| `flood_risk_zones_4tier.tif` | 4-class raster | SAFE/WATCH/WARNING/EMERGENCY per pixel |
| `flood_risk_zones.geojson` | Vector polygons | Web-ready risk zones for mapping APIs |
| `floodwatch_ghana_alert_payload.json` | JSON | Direct feed for FloodWatch Ghana alerting system |
| `flood_classifier_accra.onnx` | ONNX model | Deployable inference on new SAR acquisitions |

### Why SAR first, optical second

Accra's rainy season is characterised by dense cloud cover — exactly the conditions
where optical satellites become blind. The June 2015 disaster happened at night,
during a storm. Sentinel-1 SAR sees through cloud and darkness at 6-day repeat.
An operational system built on optical data would have been unable to map that
flood in time. SAR is not a backup — it is the primary sensor for active flood events.

### Upgrade path to operational

1. **Replace pseudo-labels with Sen1Floods11** using `SARPrithviAdapter(mode='fine_tune')` + TerraTorch — expected F1 improvement from ~0.75 to ~0.92.
2. **Add IoT sensor fusion**: pair with FloodWatch Ghana ESP32 water-level readings to provide the combined sensor+satellite confirmation logic that prevents false alarms.
3. **6-day automation**: schedule this notebook to run automatically on every new Sentinel-1 descending pass over Accra (use `asf_search` to monitor).
4. **DEM integration**: add SRTM/CopernicusDEM elevation to clip flood detection to hydrologically plausible areas (water can't flow uphill), reducing false positives in elevated urban areas.
5. **Community WhatsApp distribution**: route the `floodwatch_ghana_alert_payload.json` through FloodWatch Ghana's Twilio WhatsApp delivery pipeline.

### Scale: this notebook vs FloodWatch Ghana Phase 2

This notebook produces a one-time analysis. The FloodWatch Ghana IoT platform (the hackathon project) automates this pipeline end-to-end: sensors + satellite data → Django backend → alert rules engine → WhatsApp/SMS delivery → NADMO dashboard. This notebook is the scientific foundation; the IoT platform is the operational deployment.
